# Cohort-04 Photometry: Preprocessing, Quality Control and Outcome-Aligned Responses

This notebook takes the fibre-photometry recordings of cohort 04 (nLightG2 norepinephrine sensor in
prelimbic cortex, freely moving mice, 3x3 bandit task) from raw pyPhotometry files to
outcome-aligned responses, and checks the quality of every step.

**What it does, in order**

1. Load the recordings and the trial outcome times; count trials.
2. Check the synchronisation of photometry, video and behaviour onto **pyControl time (seconds)**.
3. Pipeline A: the order recommended in the "Lights, fiber, action!" primer (Simpson et al. 2024).
4. Pipeline B: the procedure of the nLightG2 paper (Rohner et al. 2026), as far as it could be reconstructed.
5. Compare the two in writing.
6. Quality control for every step, plus signal-to-noise (SNR) measures and a per-mouse ranking.
7. Responses to rewarded and unrewarded outcomes, per mouse.
8. A trial-matched random-time control for those responses.
9. Summary and open questions.

**Where the code lives.** Loading, synchronisation, outcome times and the cutting of event windows are
imported from `src/` (they are tested there). The photometry preprocessing and quality-control functions
are defined in this notebook, on purpose: the preprocessing is preliminary and will change, and
it will move into `src/` once it settles. Each function is still small, pure (arrays in, arrays out) and
kept apart from loading and plotting.

**Conventions.** Every time is in seconds on **pyControl's clock**. Signals are volts until
they are normalised. All parameters sit in the configuration cell below; nothing else needs editing.
Figures are also written to `results/figures/photometry/cohort-04/` and tables to
`results/tables/photometry/cohort-04/` (git-ignored).

In [ ]:
import time
import zlib
from collections.abc import Callable
from functools import partial
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import ndimage
from scipy import signal as sps
from scipy.integrate import trapezoid
from scipy.optimize import curve_fit
from scipy.stats import linregress

from src.behavior_analysis.outcome_events import load_outcome_events, outcome_settings
from src.behavior_import.import_data import read_tsv_file
from src.behavior_visualization.plot_rsync_alignment import plot_rsync_alignment
from src.photometry_analysis.event_windows import (
    extract_event_windows,
    gap_spans,
    random_event_times,
    relative_time_grid,
    sync_coverage,
)
from src.photometry_import.pyphotometry import import_ppd
from src.photometry_import.recordings import (
    MAX_MEDIAN_DISCREPANCY_MS,
    MIN_FRACTION_MATCHED,
    load_photometry_recordings,
    read_tsv_sync_pulses,
)
from src.photometry_import.sessions import find_photometry_sessions
from src.settings import REPO_ROOT, get_settings
from src.sync.pycontrol_time import photometry_sample_times, sync_quality, video_frame_times
from src.sync.rsync import RsyncError, min_pulses_required

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.float_format", "{:.3f}".format)
plt.rcParams.update(
    {
        "figure.dpi": 90,
        "savefig.dpi": 130,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "legend.fontsize": 8,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

## 1. Setup and inputs

### Configuration

Everything you may want to change is here. The values in `PREPROCESSING` are part of the cache key (see
below), so changing them recomputes the affected recordings and nothing else.

- **`PHASES`**: which trial phases enter the analysis. `task` is the bandit task proper; `port_finding` is the
  phase in which the mouse finds the ports. Both are included by default; counts per phase are reported below.
- **`EVENT_WINDOW`**: −5 to +10 s around the outcome, as in the earlier notebooks.
- **Pipeline B baselines**: Meg's earlier choice (−5 to 0 s). The nLightG2 paper's values (as adapted from
  the methods excerpt in the earlier notebook) are 20 s for the ΔF/F baseline and 15 s for the z-score baseline.
  The event windows are widened automatically if you select longer baselines.
- **QC thresholds** only *flag* a recording; nothing is excluded. They are starting points, not validated limits.

In [ ]:
# --- Data ---
COHORT = "cohort-04"
EXPERIMENT = "3x3_field_magnitude_bandit"
PHASES = ("task", "port_finding")  # trial phases to include

# --- Event windows and the random-time control ---
EVENT_WINDOW = (-5.0, 10.0)  # s relative to the outcome
RANDOM_SEED = 0
N_RANDOM_DRAWS = 200  # random draws per recording and condition, for the null distribution
# False (default): random times fall only within the recording's task period, i.e. between its
# first and last outcome event, as real outcomes do. True: anywhere in the sync coverage (the
# earlier behaviour, which also draws from the minutes before the task starts and after it ends).
RANDOM_TIMES_WHOLE_COVERAGE = False

# --- Preprocessing (shared by pipelines A and B up to dF/F). Part of the cache key. ---
PREPROCESSING = {
    "median_filter_s": None,  # window of the optional spike filter (s); None = off
    "lowpass_hz": 10.0,  # zero-phase Butterworth (order 2) cutoff
    "lowpass_order": 2,
    "fit_decimation": 13,  # the bleaching fit uses every 13th sample (10 Hz); see section 3
    "fit_initial": (0.5, 0.25, 0.25, 3600.0, 0.1),  # fractions of max for the first three; s; ratio
    "fit_lower": (
        0.0,
        0.0,
        0.0,
        600.0,
        0.0,
    ),  # const, amp_fast, amp_slow (x max); tau_slow (s); tau ratio
    "fit_upper": (1.0, 1.0, 1.0, 36000.0, 1.0),
    "code_version": 2,  # bump when a helper function used inside the cached step changes
}
HIGHPASS_ALTERNATIVE_HZ = 0.001  # comparison only, never used downstream

# --- Pipeline B: event-level normalisation (the paper's values: 20 s and 15 s) ---
B_DFF_BASELINE = (-5.0, 0.0)
B_Z_BASELINE = (-5.0, 0.0)
B_SMOOTHING_POINTS = 80  # centred moving mean; 80 points = 0.615 s at 130 Hz
METRIC_WINDOW = (0.0, 10.0)  # post-event window for peak and area under the curve

# --- Quality control ---
QC_NOISE_BAND_HZ = (
    15.0,
    50.0,
)  # noise floor band, also the high-pass cutoff for the noise estimate
QC_SIGNAL_BAND_HZ = (0.1, 3.0)  # band of slow fluctuations
QC_WELCH_SEGMENT_S = 30.0
QC_FLATLINE_S = 0.5  # identical samples for longer than this = dropout
QC_JUMP_WINDOW_S = 0.5
QC_JUMP_MAD_THRESHOLD = 10.0  # robust SD (1.4826 x MAD) units of the 0.5-s mean-difference step
QC_BASELINE_SPIKE_SD = (
    6.0  # a baseline (LED-off) sample this many robust SD above its median is a spike
)
QC_TRANSIENT_PROMINENCE_SD = 5.0  # k, in robust SD of the low-passed trace; 3 lets noise ripples in
EVOKED_POST = (0.0, 2.0)
EVOKED_PRE = (-2.0, 0.0)
EVOKED_MIN_TRIALS = 5
QC_THRESHOLDS = {
    "min_duration_min": 5.0,
    "max_baseline_v": 0.1,  # LED-off level; larger suggests ambient light leaking in
    "max_clipping_fraction": 0.001,
    "max_flatline_runs": 0,
    "min_sync_fraction": 0.95,
    "max_sync_discrepancy_samples": 2.0,
    "max_outside_sync_fraction": 0.05,
    "max_hf_noise_pct": 1.0,  # high-frequency noise SD as % of the median signal, before the low-pass
    "max_bleaching_drop_pct": 50.0,
    "max_params_at_bound": 2,
    "min_movement_slope": 0.0,
    "max_jumps_per_min": 1.0,
    "min_snr_db": 0.0,
    "min_specificity_db": 1.0,  # uncorrected sensor SNR minus control SNR
    "min_abs_evoked_z": 2.0,  # |z| of the mean evoked response against random times (significance only)
    "min_evoked_trials": EVOKED_MIN_TRIALS,
}
MIN_RANK_DISAGREEMENT = 3  # report mice whose two SNR ranks differ by at least this much

# --- Outputs ---
FIGURE_DIR = REPO_ROOT / "results" / "figures" / "photometry" / COHORT
TABLE_DIR = REPO_ROOT / "results" / "tables" / "photometry" / COHORT

# Expanded once: the window that must be extractable so every baseline and metric window fits.
EXTRACTION_WINDOW = (
    min(EVENT_WINDOW[0], B_DFF_BASELINE[0], B_Z_BASELINE[0], EVOKED_PRE[0]),
    max(EVENT_WINDOW[1], METRIC_WINDOW[1], EVOKED_POST[1]),
)

### Saving figures and tables

These two small functions are the only places that write files, so analysis and plotting code never
touches the disk.

In [ ]:
def save_figure(fig: plt.Figure, name: str) -> Path:
    """Write *fig* to the cohort's figure folder as a PNG and return the path."""
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    path = FIGURE_DIR / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    return path


def save_table(table: pd.DataFrame, name: str) -> Path:
    """Write *table* to the cohort's table folder as a CSV and return the path."""
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    path = TABLE_DIR / f"{name}.csv"
    table.to_csv(path, index=False)
    return path


def show_table(table: pd.DataFrame, name: str) -> None:
    """Print *table*, save it as CSV, and say where."""
    display(table)
    print(f"saved: {save_table(table, name).relative_to(REPO_ROOT)}")

### Find and load the recordings

`find_photometry_sessions` lists the numbered sessions that hold a `.ppd` file (`ses-test` folders are
left out). `load_photometry_recordings` pairs each `.ppd` with the pyControl `.tsv` that shares its sync
pulses and gives every photometry sample its pyControl time. A `.ppd` that cannot be paired (for example
a false start with almost no sync pulses) is returned as *skipped* with a reason, and is listed here.

In [ ]:
cohort_dir = get_settings().data_root / EXPERIMENT / "rawdata" / COHORT
sessions = find_photometry_sessions(cohort_dir)
loaded = [load_photometry_recordings(session) for session in sessions]
recordings = [rec for recs, _ in loaded for rec in recs]
skipped = [skip for _, skips in loaded for skip in skips]
MICE = list(dict.fromkeys(rec.subject for rec in recordings))


def recording_id(recording) -> str:
    """Unique label of a recording: its .ppd file name (a session can hold two recordings)."""
    return recording.ppd_path.stem


def label_outcomes(events: pd.DataFrame) -> pd.DataFrame:
    """*events* with a ``condition`` column: ``rewarded`` or ``unrewarded``."""
    events = events.copy()
    events["condition"] = np.where(events["rewarded"], "rewarded", "unrewarded")
    return events


def load_recording_behaviour(recordings: dict) -> tuple[dict, dict, list[dict]]:
    """Outcome events and logged outcome durations of each recording, with per-recording failures.

    One recording whose ``.tsv`` cannot be parsed (a ``ValueError``, which includes the parser's
    own ``OutcomeParseError``) must not abort the others, so it is reported instead of raised.

    Returns:
        tuple[dict, dict, list[dict]]: Events by recording id, logged settings by recording id,
        and one ``{"recording", "tsv", "reason"}`` row per recording that failed.
    """
    events, settings, failures = {}, {}, []
    for rid, rec in recordings.items():
        try:
            events[rid] = label_outcomes(load_outcome_events(rec.tsv_path))
            settings[rid] = outcome_settings(read_tsv_file(rec.tsv_path))
        except ValueError as error:
            failures.append({"recording": rid, "tsv": rec.tsv_path.name, "reason": str(error)})
    return events, settings, failures


RECORDINGS = {recording_id(rec): rec for rec in recordings}
assert len(RECORDINGS) == len(recordings), "recording ids must be unique"
ALL_EVENTS, EXPECTED, behaviour_failures = load_recording_behaviour(RECORDINGS)
if behaviour_failures:
    print(
        f"\nBehaviour could not be loaded for {len(behaviour_failures)} recording(s); they are left out:"
    )
    display(pd.DataFrame(behaviour_failures))
RECORDINGS = {rid: rec for rid, rec in RECORDINGS.items() if rid in ALL_EVENTS}
recordings = list(RECORDINGS.values())
MICE = list(dict.fromkeys(rec.subject for rec in recordings))
EVENTS = {
    rid: ev[ev["phase"].isin(PHASES)].reset_index(drop=True) for rid, ev in ALL_EVENTS.items()
}

print(
    f"{len(sessions)} sessions with a .ppd; {len(recordings)} recordings aligned; {len(MICE)} mice."
)
print(f"\nSkipped ({len(skipped)}):")
for skip in skipped:
    print(f"  {skip.ppd_path.relative_to(cohort_dir)}: {skip.reason}")

### Channel wiring (to be confirmed)

The recordings are pyPhotometry `2EX_1EM_pulsed`: two excitation LEDs, one emission detector, 130 Hz.
**Assumed wiring: `analog_1` is the nLightG2 signal (470 nm excitation) and `analog_2` is the isosbestic control
(405 nm).** This is taken from the earlier notebooks and has not been confirmed against the physical LED
connections. Everything below that calls a channel "signal" or "control" depends on it; the
movement-correction and specificity QC would be meaningless if the channels were swapped. The LED currents
(printed below) differ between the two channels and between sessions, which is worth knowing when
comparing absolute voltages.

### Recordings and trial counts

In [ ]:
def recordings_table(recordings) -> pd.DataFrame:
    """One row per aligned recording: identity, files, duration and LED currents."""
    return pd.DataFrame(
        [
            {
                "recording": recording_id(rec),
                "mouse": rec.subject,
                "session": rec.session,
                "ppd": rec.ppd_path.name,
                "tsv": rec.tsv_path.name,
                "fs_hz": rec.fs,
                "duration_min": len(rec.signal) / rec.fs / 60,
                "led_current_ma_1": rec.led_current[0],
                "led_current_ma_2": rec.led_current[1],
            }
            for rec in recordings
        ]
    )


COUNT_COLUMNS = [f"{p}_{c}" for p in ("task", "port_finding") for c in ("rewarded", "unrewarded")]


def trial_counts_table(events_by_recording: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Trials per recording by phase and outcome (columns ``<phase>_<condition>``)."""
    rows = []
    for rid, events in events_by_recording.items():
        rec = RECORDINGS[rid]
        counts = events.groupby(["phase", "condition"]).size()
        row = {"recording": rid, "mouse": rec.subject, "session": rec.session}
        for column in COUNT_COLUMNS:
            phase, condition = column.rsplit("_", 1)
            row[column] = int(counts.get((phase, condition), 0))
        rows.append(row)
    table = pd.DataFrame(rows)
    table["total"] = table[COUNT_COLUMNS].sum(axis=1)
    return table


recordings_summary = recordings_table(recordings)
show_table(recordings_summary, "recordings")

trial_counts = trial_counts_table(ALL_EVENTS)
show_table(trial_counts, "trial_counts")
phase_totals = trial_counts[COUNT_COLUMNS].sum()
print("Trials over all recordings, by phase and outcome (all phases counted here):")
print(phase_totals.to_string())
print(f"\nIncluded by PHASES={PHASES}: {sum(len(e) for e in EVENTS.values())} trials.")

## 2. Synchronisation checks

Photometry, video and behaviour each run on their own clock. pyControl sends random-interval `rsync` pulses to
the other systems; `RsyncAligner` finds which pulse is which by matching the *pattern* of intervals, and
the time between pulses is interpolated. Photometry samples before the first and after the last matched pulse get
no time (NaN), and no window is allowed to reach into them.

**What to look for.** `fraction_matched` close to 1; the largest interval disagreement between the two clocks
within about one photometry sample (7.7 ms at 130 Hz); a few percent of samples outside the sync coverage
(the recording starts before, and ends after, the first and last pulse).

In [ ]:
def sync_table(recordings) -> pd.DataFrame:
    """One row per recording: how well its sync pulses matched pyControl's."""
    rows = []
    for rec in recordings:
        quality = rec.sync_quality
        sample_ms = 1000 / rec.fs
        rows.append(
            {
                "recording": recording_id(rec),
                "mouse": rec.subject,
                "session": rec.session,
                "n_pulses_ppd": quality["n_pulses_other"],
                "n_pulses_pycontrol": quality["n_pulses_pycontrol"],
                "n_matched": quality["n_matched"],
                "fraction_matched": quality["fraction_matched"],
                "max_discrepancy_ms": quality["max_abs_interval_discrepancy_ms"],
                "max_discrepancy_samples": quality["max_abs_interval_discrepancy_ms"] / sample_ms,
                "median_discrepancy_ms": quality["median_abs_interval_discrepancy_ms"],
                "first_matched_s": quality["first_matched_time_s"],
                "last_matched_s": quality["last_matched_time_s"],
                "n_samples_outside_sync": rec.n_samples_outside_sync,
                "fraction_outside_sync": rec.n_samples_outside_sync / len(rec.signal),
            }
        )
    return pd.DataFrame(rows)


sync_summary = sync_table(recordings)
show_table(sync_summary, "sync_quality")

### Alignment diagnostic for one example recording

The example is the recording with the most trials (used again for the step-by-step plots below). The left two
panels should show two cleanly separated groups of chunk errors and interval disagreements of a few ms at most;
the wide panel should be a straight line.

In [ ]:
EXAMPLE = max(EVENTS, key=lambda rid: len(EVENTS[rid]))
print(f"Example recording: {EXAMPLE} ({len(EVENTS[EXAMPLE])} trials)")


def recording_aligner(recording):
    """The rsync aligner of a recording (reloads the .ppd pulses and the .tsv rsync times)."""
    data = import_ppd(recording.ppd_path, low_pass=None, high_pass=None)
    pulses = read_tsv_sync_pulses(recording.tsv_path)
    return photometry_sample_times(data, pulses)[1]


example_aligner = recording_aligner(RECORDINGS[EXAMPLE])
axes = plot_rsync_alignment(example_aligner)
axes[0].figure.suptitle(f"Photometry-To-pyControl Sync: {EXAMPLE}", fontsize=11)
save_figure(axes[0].figure, "sync_alignment_example")
plt.show()

### Video on the same clock

The camera writes a `*_GPIO_data.csv` with one row per frame and the rsync pulses on `GPIO1`. The same aligner
puts video frame times on pyControl time, which is what makes frame-accurate behaviour-photometry comparisons
possible. A session can hold several `.tsv` files, so each video is matched to the `.tsv` whose pulse train it shares.

In [ ]:
def match_video_to_tsv(gpio_df: pd.DataFrame, tsv_pulses: dict[Path, np.ndarray]):
    """The one candidate ``.tsv`` whose rsync train aligns with the video's, or None.

    Returns:
        tuple | None: ``(tsv_path, frame_times_s, aligner)``. The acceptance rule is the one used to
        pair a ``.ppd`` with a ``.tsv``: most pulses paired, and a small median interval error.

    Raises:
        ValueError: If more than one ``.tsv`` matches (the pairing would be ambiguous), as in
            ``match_ppd_to_tsv``.
    """
    matches = []
    for path, pulses in tsv_pulses.items():
        if len(pulses) < min_pulses_required():
            continue
        try:
            frame_times, aligner = video_frame_times(gpio_df, pulses)
        except RsyncError:  # not the right .tsv for this video
            continue
        quality = sync_quality(aligner)
        if (
            quality["fraction_matched"] >= MIN_FRACTION_MATCHED
            and quality["median_abs_interval_discrepancy_ms"] <= MAX_MEDIAN_DISCREPANCY_MS
        ):
            matches.append((path, frame_times, aligner))
    if len(matches) > 1:
        raise ValueError(f"the video matches more than one .tsv: {[m[0].name for m in matches]}.")
    return matches[0] if matches else None


def video_sync_row(session, gpio_path: Path, match) -> dict:
    """One row of the video sync table; *match* is the result of :func:`match_video_to_tsv`."""
    row = {"mouse": session.subject, "session": session.session, "gpio": gpio_path.name}
    if match is None:
        return {**row, "tsv": "no matching .tsv"}
    tsv_path, frame_times, aligner = match
    quality = sync_quality(aligner)
    covered = np.isfinite(frame_times)
    return {
        **row,
        "tsv": tsv_path.name,
        "n_frames": len(frame_times),
        "frame_rate_hz": 1 / np.median(np.diff(frame_times[covered])),
        "fraction_matched": quality["fraction_matched"],
        "max_discrepancy_ms": quality["max_abs_interval_discrepancy_ms"],
        "median_discrepancy_ms": quality["median_abs_interval_discrepancy_ms"],
        "fraction_outside_sync": 1 - covered.mean(),
    }


video_rows, video_aligners = [], {}
for session in sessions:
    if not session.gpio_paths:
        continue
    tsv_pulses = {path: read_tsv_sync_pulses(path) for path in session.tsv_paths}
    for gpio_path in session.gpio_paths:
        match = match_video_to_tsv(pd.read_csv(gpio_path), tsv_pulses)
        video_rows.append(video_sync_row(session, gpio_path, match))
        if match is not None:
            video_aligners[(session.subject, session.session, gpio_path.name)] = match[2]
video_sync = pd.DataFrame(video_rows)
show_table(video_sync, "video_sync_quality")
print(f"{len(sessions) - sum(1 for s in sessions if s.gpio_paths)} sessions have no video.")

In [ ]:
if video_aligners:
    video_example = next(iter(video_aligners.items()))
    axes = plot_rsync_alignment(video_example[1])
    axes[0].figure.suptitle(
        "Video-To-pyControl Sync: " + " ".join(video_example[0][:2]), fontsize=11
    )
    save_figure(axes[0].figure, "sync_alignment_video_example")
    plt.show()

### Outcome timing check

Event times come from the pyControl log (`load_outcome_events`):

- **Rewarded trial**: zero is entry to the `choice` state, which is the start of reward pulse 1. The first
  `pulse_off_timer` after it ends the pulse, so `check_time - outcome_time` should be about `pulse_isi`.
- **Unrewarded trial**: zero is entry to `ITI`, which is the onset of the no-reward cue. `no_rew_cue_timer` fires
  at cue *offset*, so `check_time - outcome_time` should be about `no_rew_cue_dur`.

The expected values are read from each session's `run_start` row. **What to look for:** narrow peaks at the
expected values (the dashed lines).

The measured delays are about 2 ms longer than `pulse_isi` (302 vs 300 ms) and about 6 ms shorter than
`no_rew_cue_dur` (494 vs 500 ms). The cause of the 6 ms has not been established here (it needs a look at the
task code); what matters for the photometry is its size. One photometry sample lasts 7.7 ms at 130 Hz, so a 6 ms
offset in the cue onset is below the sampling resolution and cannot move a response by more than one sample.
Whether to correct it anyway is an open question in the summary.

In [ ]:
def outcome_delays(events: pd.DataFrame) -> pd.DataFrame:
    """Delay (ms) from outcome to check time, with the outcome kind."""
    return pd.DataFrame(
        {
            "kind": events["outcome_kind"],
            "delay_ms": 1000 * (events["check_time"] - events["outcome_time"]),
        }
    )


def timing_summary(delays: dict[str, pd.DataFrame], expected: dict[str, dict]) -> pd.DataFrame:
    """Per recording and outcome kind: median delay against the logged duration."""
    rows = []
    for rid, table in delays.items():
        for kind, setting in (
            ("reward_onset", "pulse_isi"),
            ("no_reward_cue_onset", "no_rew_cue_dur"),
        ):
            values = table.loc[table["kind"] == kind, "delay_ms"]
            if values.empty:
                continue
            rows.append(
                {
                    "recording": rid,
                    "kind": kind,
                    "n": len(values),
                    "expected_ms": expected[rid][setting],
                    "median_ms": values.median(),
                    "p5_ms": values.quantile(0.05),
                    "p95_ms": values.quantile(0.95),
                    "median_minus_expected_ms": values.median() - expected[rid][setting],
                }
            )
    return pd.DataFrame(rows)


def plot_delay_histogram(ax, delays_ms: np.ndarray, expected_ms: float, title: str) -> None:
    """Histogram of check-time delays with the logged duration marked."""
    ax.hist(delays_ms, bins=np.arange(expected_ms - 20, expected_ms + 20.5, 1.0), color="0.5")
    ax.axvline(expected_ms, color="k", ls="--", label=f"Logged Duration ({expected_ms:g} ms)")
    ax.axvline(
        np.median(delays_ms), color="tab:red", label=f"Median ({np.median(delays_ms):.1f} ms)"
    )
    ax.set(title=title, xlabel="Check Time Minus Outcome Time (ms)", ylabel="Trials")
    ax.legend()


DELAYS = {rid: outcome_delays(ev) for rid, ev in ALL_EVENTS.items()}
timing = timing_summary(DELAYS, EXPECTED)
show_table(timing, "outcome_timing")

all_delays = pd.concat(DELAYS.values())
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), constrained_layout=True)
for ax, (kind, setting, title) in zip(
    axes,
    [
        ("reward_onset", "pulse_isi", "Rewarded: Reward Pulse 1"),
        ("no_reward_cue_onset", "no_rew_cue_dur", "Unrewarded: No-Reward Cue"),
    ],
    strict=True,
):
    logged = np.unique([e[setting] for e in EXPECTED.values()])
    assert len(logged) == 1, f"{setting} differs between sessions: {logged}"
    expected_ms = float(logged[0])
    plot_delay_histogram(
        ax, all_delays.loc[all_delays["kind"] == kind, "delay_ms"].to_numpy(), expected_ms, title
    )
save_figure(fig, "outcome_timing")
plt.show()

## 3. Pipeline A: the "Lights, fiber, action!" primer

Simpson, Akam, Patriarchi et al. (2024), *Neuron*, "Lights, fiber, action! A primer on in vivo fiber photometry"
([PMC10939905](https://pmc.ncbi.nlm.nih.gov/articles/PMC10939905)). The primer's order is
**filtering, then bleaching correction, then movement correction, then normalisation**, and the bleaching
correction is done on each channel *before and independently of* the movement correction. Each step below is one
small function:

| Step | What it does | Default | Why |
|---|---|---|---|
| A1 | LED-off baseline subtracted (done by the importer) | on | removes ambient light and electronic offset from each sample |
| A2 | Median filter for electrical spikes | **off** | needed only if there are brief spikes; window about 5x the spike duration |
| A3 | Zero-phase Butterworth low-pass (order 2, `filtfilt`) | 10 Hz | removes high-frequency noise without shifting events in time. The primer gives 2-10 Hz as typical for GCaMP6f/dLight; nLightG2 is not known to be faster than these, so the top of that range is used |
| A4 | Bleaching: double-exponential fit per channel, subtracted | on | the primer's compromise between a single exponential (too rigid) and a high-pass filter (removes real slow signal) |
| A5 | Movement: least-squares fit of the detrended control onto the detrended signal, subtracted | on | what is shared between the channels (movement, light-path changes) is not sensor activity |
| A6 | Normalisation: ΔF/F (%) = 100 x corrected / signal bleaching fit; session z-score of the corrected signal | both | makes sessions and mice comparable |

**Subtract or divide for bleaching?** Subtracting the fit assumes the bleaching comes from a constant
background (autofluorescence of the fibre and tissue) that is not modulated by the indicator; dividing assumes the
bleaching is of the indicator itself, so the signal scales with it. Autofluorescence-dominated bleaching
favours subtraction, indicator-dominated bleaching favours division. Here the fit is **subtracted** (as in the
earlier notebooks) and the fit is then used as the denominator of ΔF/F, which is the same choice
the earlier notebook made. Which assumption holds for these fibres is not tested here.

**A high-pass filter** (0.001 Hz) is also computed for one example below, for comparison only: it needs no
model, but also removes genuine slow changes in the signal.

**The bleaching fit** keeps the parameterisation and bounds of the earlier notebook (constant, fast and slow
amplitudes, slow time constant 600-36000 s, fast time constant as a fraction of the slow one). It is fitted to every
13th sample (10 Hz) and the curve is evaluated at every sample: the fit has only slow timescales, a full-rate fit
costs about 5 s per channel, and the two agree to well under 1% (checked on synthetic data below).

In [ ]:
PARAMETER_NAMES = ("const", "amp_fast", "amp_slow", "tau_slow", "tau_multiplier")


def median_despike(x: np.ndarray, fs: float, window_s: float | None) -> np.ndarray:
    """Step A2: running median over *window_s* seconds; returns *x* unchanged if None.

    The window is rounded up to an odd number of samples and the edges are padded with the nearest
    value, so no artificial step appears at the ends.
    """
    if window_s is None:
        return x
    size = int(round(window_s * fs))
    size += 1 - size % 2
    return ndimage.median_filter(x, size=size, mode="nearest")


def lowpass_zero_phase(x: np.ndarray, fs: float, cutoff_hz: float, order: int = 2) -> np.ndarray:
    """Step A3: Butterworth low-pass applied forwards and backwards, so events are not shifted in time."""
    if not 0 < cutoff_hz < fs / 2:
        raise ValueError(
            f"cutoff {cutoff_hz} Hz must lie between 0 and the Nyquist frequency {fs / 2} Hz."
        )
    b, a = sps.butter(order, cutoff_hz, btype="low", fs=fs)
    return sps.filtfilt(b, a, x)


def highpass_zero_phase(x: np.ndarray, fs: float, cutoff_hz: float, order: int = 2) -> np.ndarray:
    """Zero-phase Butterworth high-pass (second-order sections, stable at very low cutoffs)."""
    sos = sps.butter(order, cutoff_hz, btype="high", fs=fs, output="sos")
    return sps.sosfiltfilt(sos, x, padtype="even")


def double_exponential(t, const, amp_fast, amp_slow, tau_slow, tau_multiplier):
    """Constant plus slow and fast decaying exponentials; the fast time constant is ``tau_slow * tau_multiplier``."""
    return (
        const
        + amp_slow * np.exp(-t / tau_slow)
        + amp_fast * np.exp(-t / (tau_slow * tau_multiplier))
    )


def fit_bleaching(
    t: np.ndarray, y: np.ndarray, settings: dict
) -> tuple[np.ndarray, np.ndarray, tuple]:
    """Step A4 (fit): double-exponential bleaching curve for one channel.

    Args:
        t (np.ndarray): Time from the start of the recording (s).
        y (np.ndarray): The channel, low-passed (V).
        settings (dict): ``PREPROCESSING``: uses ``fit_decimation``, ``fit_initial``, ``fit_lower``,
            ``fit_upper``. The first three entries of each are fractions of ``max(y)``.

    Returns:
        tuple[np.ndarray, np.ndarray, tuple]: The curve at every sample (V), the fitted parameters
        (see ``PARAMETER_NAMES``), and the (lower, upper) bounds used.

    Raises:
        RuntimeError: If the fit does not converge in 1000 evaluations.
        ValueError: If the curve is not positive everywhere, which would break ΔF/F.
    """
    scale = np.array([y.max()] * 3 + [1.0, 1.0])
    bounds = (np.array(settings["fit_lower"]) * scale, np.array(settings["fit_upper"]) * scale)
    step = settings["fit_decimation"]
    params, _ = curve_fit(
        double_exponential,
        t[::step],
        y[::step],
        p0=np.array(settings["fit_initial"]) * scale,
        bounds=bounds,
        maxfev=1000,
    )
    curve = double_exponential(t, *params)
    if not np.all(curve > 0):
        raise ValueError("the bleaching fit is not positive everywhere.")
    return curve, params, bounds


def regress_out_control(signal_detrended: np.ndarray, control_detrended: np.ndarray):
    """Step A5: subtract the least-squares fit of the control from the signal.

    Returns:
        tuple[np.ndarray, np.ndarray, dict]: The corrected signal, the estimated movement component, and
        ``slope``, ``intercept``, ``r`` (Pearson) and ``r2`` of the regression of signal on control.
    """
    fit = linregress(control_detrended, signal_detrended)
    movement = fit.intercept + fit.slope * control_detrended
    stats = {"slope": fit.slope, "intercept": fit.intercept, "r": fit.rvalue, "r2": fit.rvalue**2}
    return signal_detrended - movement, movement, stats


def delta_f_over_f(corrected: np.ndarray, baseline_fit: np.ndarray) -> np.ndarray:
    """Step A6: percent change relative to the (time-varying) bleaching fit."""
    return 100 * corrected / baseline_fit


def session_zscore(x: np.ndarray) -> np.ndarray:
    """Step A6: subtract the session mean and divide by the session standard deviation."""
    sd = x.std()
    if not sd > 0:
        raise ValueError("cannot z-score a constant trace.")
    return (x - x.mean()) / sd


def preprocess_channels(
    signal: np.ndarray, control: np.ndarray, fs: float, settings: dict, lowpass: bool = True
) -> dict:
    """Steps A2-A6 for one recording; every intermediate is returned so any stage can be inspected.

    Args:
        signal, control (np.ndarray): Baseline-subtracted channels (V), same length.
        fs (float): Sampling rate (Hz).
        settings (dict): ``PREPROCESSING``.
        lowpass (bool): Apply step A3. False gives the unfiltered movement-corrected ΔF/F that the
            spectral SNR needs (a 10 Hz low-pass would hide the noise floor above 15 Hz).

    Returns:
        dict: ``denoised_signal/control`` (V), ``fit_signal/control`` (V), ``params_signal/control``,
        ``bounds_signal/control``, ``detrended_signal/control`` (V), ``movement`` (V), ``corrected`` (V),
        ``motion_stats`` (dict), ``dff`` (%), ``uncorrected_dff`` (%, before step A5), ``control_dff`` (%, the
        control's own ΔF/F) and ``zscore``.
    """
    if signal.shape != control.shape or not (
        np.isfinite(signal).all() and np.isfinite(control).all()
    ):
        raise ValueError("signal and control must have the same length and be finite.")
    t = np.arange(len(signal)) / fs
    channels = {}
    for name, raw in (("signal", signal), ("control", control)):
        x = median_despike(raw, fs, settings["median_filter_s"])
        if lowpass:
            x = lowpass_zero_phase(x, fs, settings["lowpass_hz"], settings["lowpass_order"])
        fit, params, bounds = fit_bleaching(t, x, settings)
        channels[name] = {
            "denoised": x,
            "fit": fit,
            "params": params,
            "bounds": bounds,
            "detrended": x - fit,
        }
    corrected, movement, stats = regress_out_control(
        channels["signal"]["detrended"], channels["control"]["detrended"]
    )
    out = {"movement": movement, "corrected": corrected, "motion_stats": stats}
    for name, parts in channels.items():
        for key, value in parts.items():
            out[f"{key}_{name}"] = value
    out["dff"] = delta_f_over_f(corrected, channels["signal"]["fit"])
    out["control_dff"] = delta_f_over_f(
        channels["control"]["detrended"], channels["control"]["fit"]
    )
    out["uncorrected_dff"] = delta_f_over_f(
        channels["signal"]["detrended"], channels["signal"]["fit"]
    )
    out["zscore"] = session_zscore(corrected)
    return out

### Synthetic check of pipeline A

Before touching real data, the pipeline is run on a recording with a **known** answer: a double-exponential
bleaching curve per channel, a movement artefact that enters the signal with gain 1.0 and the control with
gain 0.8 (so the true regression slope of signal on control is 1.25), a few transients in the signal only,
and white noise. The pipeline should recover the bleaching curve, the slope, and the transients.

In [ ]:
def band_limited_noise(rng, n: int, fs: float, band: tuple[float, float], sd: float) -> np.ndarray:
    """Gaussian noise confined to *band* (Hz), scaled to standard deviation *sd*."""
    spectrum = np.fft.rfft(rng.standard_normal(n))
    freqs = np.fft.rfftfreq(n, 1 / fs)
    spectrum[(freqs < band[0]) | (freqs > band[1])] = 0
    x = np.fft.irfft(spectrum, n)
    return sd * x / x.std()


def transient_shape(
    t: np.ndarray, onset: float, rise_s: float = 0.15, decay_s: float = 1.0
) -> np.ndarray:
    """Unit-peak transient: fast rise, slower exponential decay, starting at *onset*."""
    dt = np.clip(t - onset, 0, None)
    shape = (1 - np.exp(-dt / rise_s)) * np.exp(-dt / decay_s)
    return shape / shape.max()


def synthetic_recording(
    rng, fs=130.0, minutes=15.0, noise_sd=0.002, n_transients=20, amplitude=0.03
):
    """Signal and control with a known bleaching curve, shared movement and signal-only transients.

    Returns:
        tuple[np.ndarray, np.ndarray, dict]: signal (V), control (V), and the truth: ``fit_signal``,
        ``fit_control``, ``slope`` (true signal-on-control slope), ``transients`` (signal-only part) and
        ``onsets`` (s).
    """
    n = int(minutes * 60 * fs)
    t = np.arange(n) / fs
    fit_signal = 1.2 + 0.9 * np.exp(-t / 3000) + 0.4 * np.exp(-t / 300)
    fit_control = 1.0 + 0.7 * np.exp(-t / 3000) + 0.3 * np.exp(-t / 300)
    movement = band_limited_noise(rng, n, fs, (0.1, 4.0), sd=0.02)
    onsets = np.sort(rng.uniform(20, minutes * 60 - 20, n_transients))
    transients = amplitude * sum(transient_shape(t, onset) for onset in onsets)
    signal = fit_signal + movement + transients + noise_sd * rng.standard_normal(n)
    control = fit_control + 0.8 * movement + noise_sd * rng.standard_normal(n)
    truth = {
        "fit_signal": fit_signal,
        "fit_control": fit_control,
        "slope": 1 / 0.8,
        "transients": transients,
        "onsets": onsets,
    }
    return signal, control, truth


def check_pipeline_a_on_synthetic_data(settings: dict) -> pd.DataFrame:
    """Run pipeline A on a synthetic recording and assert that it recovers the truth."""
    fs = 130.0
    signal, control, truth = synthetic_recording(np.random.default_rng(1), fs)
    out = preprocess_channels(signal, control, fs, settings)
    bleaching_error = np.abs(out["fit_signal"] / truth["fit_signal"] - 1).max()
    slope_error = abs(out["motion_stats"]["slope"] / truth["slope"] - 1)
    expected = lowpass_zero_phase(
        truth["transients"], fs, settings["lowpass_hz"], settings["lowpass_order"]
    )
    transient_r = np.corrcoef(out["corrected"], expected)[0, 1]
    # full-rate fit for comparison with the decimated one
    full_rate = {**settings, "fit_decimation": 1}
    full_fit, _, _ = fit_bleaching(np.arange(len(signal)) / fs, out["denoised_signal"], full_rate)
    decimation_error = np.abs(out["fit_signal"] / full_fit - 1).max()
    assert bleaching_error < 0.02, bleaching_error
    assert slope_error < 0.10, slope_error
    assert transient_r > 0.95, transient_r
    assert decimation_error < 0.01, decimation_error
    assert np.isclose(out["zscore"].mean(), 0, atol=1e-9) and np.isclose(out["zscore"].std(), 1)
    return pd.DataFrame(
        {
            "check": [
                "Bleaching fit: largest relative error",
                "Movement slope: relative error",
                "Corrected signal vs true transients: r",
                "Decimated vs full-rate fit: largest relative difference",
            ],
            "value": [bleaching_error, slope_error, transient_r, decimation_error],
            "limit": ["< 0.02", "< 0.10", "> 0.95", "< 0.01"],
        }
    )


display(check_pipeline_a_on_synthetic_data(PREPROCESSING))


def check_median_despike() -> None:
    """The optional spike filter removes a single-sample spike and leaves a slow ramp alone."""
    ramp = np.linspace(0, 1, 500)
    spiked = ramp.copy()
    spiked[250] += 10
    cleaned = median_despike(spiked, fs=100.0, window_s=0.05)  # 5 samples
    assert np.abs(cleaned - ramp).max() < 0.01
    assert median_despike(spiked, 100.0, None) is spiked


check_median_despike()
print(
    "Median filter check passed (it removes a one-sample spike and is a no-op when switched off)."
)

### Cache the per-recording preprocessing

Fitting four exponentials and filtering per recording is the slow part of this notebook. Results are cached to
disk with `joblib.Memory`, **keyed on the file path, file size, modification time and the `PREPROCESSING`
dictionary**, so editing a parameter or replacing a file recomputes exactly that recording, and deleting
the cache folder only costs time. The loaded recording is passed along but excluded from the key (it
is fully determined by the file). The cache lives in `get_settings().cache_dir / "photometry"` and is git-ignored.

In [ ]:
memory = joblib.Memory(get_settings().cache_dir / "photometry", verbose=0)


def file_key(path: Path) -> tuple[str, int, int]:
    """Identity of a file for caching: path, size in bytes and modification time in ns."""
    info = Path(path).stat()
    return str(path), info.st_size, info.st_mtime_ns


@memory.cache(ignore=["recording"])
def preprocess_recording_cached(key: tuple, settings: dict, recording) -> dict:
    """Pipeline A (steps A2-A6) for one recording, with and without the low-pass.

    *key* and *settings* identify the computation for the cache; *recording* supplies the data.
    The unfiltered run keeps only the traces the spectral SNR needs.
    """
    filtered = preprocess_channels(recording.signal, recording.control, recording.fs, settings)
    unfiltered = preprocess_channels(
        recording.signal, recording.control, recording.fs, settings, lowpass=False
    )
    return {
        "filtered": filtered,
        "unfiltered": {
            k: unfiltered[k] for k in ("dff", "uncorrected_dff", "control_dff", "motion_stats")
        },
    }


def preprocess_recording(recording) -> dict:
    """Cached pipeline A for *recording*."""
    return preprocess_recording_cached(file_key(recording.ppd_path), PREPROCESSING, recording)

### Run pipeline A on every recording

This is the cached step; the first run takes a few minutes, later runs seconds.

In [ ]:
_start = time.time()
PROCESSED = {}
for rid, rec in RECORDINGS.items():
    PROCESSED[rid] = preprocess_recording(rec)
print(f"Pipeline A on {len(PROCESSED)} recordings: {time.time() - _start:.0f} s.")

### Every stage for one example recording

The example is the same recording as in section 2. Each figure has the whole session on the left and a 60-second
zoom on the right (the zoom is centred on the rewarded outcome closest to the middle of the session; ticks
on the last figure mark outcomes). Panels in a column share their time axis, which is pyControl time. Colours:
green is the nLightG2 signal (470 nm, assumed), purple the isosbestic control (405 nm, assumed).

In [ ]:
SIGNAL_COLOR, CONTROL_COLOR = "tab:green", "tab:purple"
Line = tuple[str, np.ndarray, dict]  # label, y, matplotlib line keywords


def zoom_window(recording, events: pd.DataFrame, length_s: float = 60.0) -> tuple[float, float]:
    """A *length_s* window centred on the rewarded outcome nearest the middle of the sync coverage."""
    first, last = sync_coverage(recording.time)
    rewarded = events.loc[events["condition"] == "rewarded", "outcome_time"].to_numpy()
    centre = rewarded[np.argmin(np.abs(rewarded - (first + last) / 2))]
    start = float(np.clip(centre - length_s / 2, first, last - length_s))
    return start, start + length_s


def plot_lines(ax, time: np.ndarray, lines: list[Line], ylabel: str, legend: bool = True) -> None:
    """Draw labelled traces against *time* on *ax*."""
    for label, y, style in lines:
        ax.plot(time, y, label=label, lw=0.6, **style)
    ax.set_ylabel(ylabel)
    if legend:
        ax.legend(loc="upper right", ncol=len(lines), framealpha=0.85)


def stage_figure(time, rows, zoom, title, outcome_times=None, axes=None):
    """One figure per processing stage: each row is ``(ylabel, lines)``; whole session | zoom.

    Panels in a column share their x-axis. *outcome_times* (optional, dict of colour -> times) adds ticks
    for outcomes along the bottom of the zoom column. *axes* is an optional ``(len(rows), 2)`` array of
    axes to draw on; if None a new figure is created. Returns the figure drawn on.
    """
    if axes is None:
        _, axes = plt.subplots(
            len(rows), 2, figsize=(13, 1.9 * len(rows) + 0.8), sharex="col", constrained_layout=True
        )
    axes = np.asarray(axes).reshape(len(rows), 2)
    fig = axes[0, 0].figure
    first, last = sync_coverage(time)
    for (ylabel, lines), (ax_full, ax_zoom) in zip(rows, axes, strict=True):
        plot_lines(ax_full, time, lines, ylabel, legend=True)
        plot_lines(ax_zoom, time, lines, ylabel, legend=False)
    axes[0, 0].set_xlim(first, last)
    axes[0, 1].set_xlim(*zoom)
    axes[0, 0].set_title("Whole Session")
    axes[0, 1].set_title("60-s Zoom")
    for ax in axes[-1]:
        ax.set_xlabel("pyControl Time (s)")
    if outcome_times:
        for colour, times in outcome_times.items():
            inside = times[(times >= zoom[0]) & (times <= zoom[1])]
            axes[-1, 1].plot(
                inside, np.full(len(inside), axes[-1, 1].get_ylim()[0]), "|", color=colour, ms=12
            )
    fig.suptitle(title, fontsize=12)
    return fig


EXAMPLE_REC, EXAMPLE_PROC = RECORDINGS[EXAMPLE], PROCESSED[EXAMPLE]["filtered"]
EXAMPLE_TIME = EXAMPLE_REC.time
EXAMPLE_ZOOM = zoom_window(EXAMPLE_REC, EVENTS[EXAMPLE])
EXAMPLE_OUTCOMES = {
    "tab:green": EVENTS[EXAMPLE].query("condition == 'rewarded'")["outcome_time"].to_numpy(),
    "tab:red": EVENTS[EXAMPLE].query("condition == 'unrewarded'")["outcome_time"].to_numpy(),
}
print(f"Zoom window: {EXAMPLE_ZOOM[0]:.0f} to {EXAMPLE_ZOOM[1]:.0f} s")

#### A1: LED-off baseline subtraction

pyPhotometry in pulsed mode records each channel twice per cycle: with the LED on, and with it off. The
importer subtracts the LED-off sample from the LED-on sample, which removes ambient light and electronic offset.
**Look for:** a LED-off baseline that is small (tens of mV) and flat compared with the LED-on level; the difference
should follow the LED-on trace closely, shifted down by the baseline. In the example the baselines also show brief upward
spikes of a few mV in both channels; they subtract as small downward blips. Their origin is unknown (they are not at the
rsync pulses) and they are counted per recording in the acquisition QC (section 6.1).

In [ ]:
fig = stage_figure(
    EXAMPLE_TIME,
    [
        (
            "Signal (V)",
            [
                ("LED on", EXAMPLE_REC.signal_led_on, {"color": "0.6"}),
                ("On minus off", EXAMPLE_REC.signal, {"color": SIGNAL_COLOR}),
            ],
        ),
        ("Signal LED off (V)", [("LED-off baseline", EXAMPLE_REC.signal_baseline, {"color": "k"})]),
        (
            "Control (V)",
            [
                ("LED on", EXAMPLE_REC.control_led_on, {"color": "0.6"}),
                ("On minus off", EXAMPLE_REC.control, {"color": CONTROL_COLOR}),
            ],
        ),
        (
            "Control LED off (V)",
            [("LED-off baseline", EXAMPLE_REC.control_baseline, {"color": "k"})],
        ),
    ],
    EXAMPLE_ZOOM,
    f"A1: LED-Off Baseline Subtraction ({EXAMPLE})",
)
save_figure(fig, "example_A1_baseline_subtraction")
plt.show()

#### A2: optional median filter

Off by default (`median_filter_s=None`). The raw traces do contain a few isolated single-sample dips that hit both channels at the same sample (for example near 745 s in the example); the regression in A5 removes shared artefacts of that kind, so the filter is left off. A running median is a
non-linear filter that removes brief single-sample spikes without blurring edges the way a low-pass does. If you
need it, set `median_filter_s` to about five times the spike duration (for example 0.04 s for 8-ms spikes). It is
applied before the low-pass, because a low-pass would smear a spike into a bump that a median can no longer
remove. The synthetic check above shows it removing a one-sample spike.

#### A3: low-pass filter (10 Hz)

**Look for:** the high-frequency fuzz of the raw trace is gone while the slow structure is unchanged.

In [ ]:
fig = stage_figure(
    EXAMPLE_TIME,
    [
        (
            "Signal (V)",
            [
                ("Before", EXAMPLE_REC.signal, {"color": "0.6"}),
                ("Low-passed 10 Hz", EXAMPLE_PROC["denoised_signal"], {"color": SIGNAL_COLOR}),
            ],
        ),
        (
            "Control (V)",
            [
                ("Before", EXAMPLE_REC.control, {"color": "0.6"}),
                ("Low-passed 10 Hz", EXAMPLE_PROC["denoised_control"], {"color": CONTROL_COLOR}),
            ],
        ),
    ],
    EXAMPLE_ZOOM,
    f"A3: Zero-Phase Low-Pass Filter ({EXAMPLE})",
)
save_figure(fig, "example_A3_lowpass")
plt.show()

#### A4: bleaching correction

A double exponential is fitted to each low-passed channel and subtracted. The high-pass alternative (0.001 Hz,
dashed) is shown on the detrended rows for comparison only; it does not feed anything downstream.
**Look for:** a smooth fit through the middle of the data without following fast fluctuations, and a
detrended trace that fluctuates around zero without a remaining slope. The two methods should agree except
for very slow components.

In [ ]:
t_example = np.arange(len(EXAMPLE_REC.signal)) / EXAMPLE_REC.fs
highpass = {
    name: highpass_zero_phase(
        EXAMPLE_PROC[f"denoised_{name}"], EXAMPLE_REC.fs, HIGHPASS_ALTERNATIVE_HZ
    )
    for name in ("signal", "control")
}
fig = stage_figure(
    EXAMPLE_TIME,
    [
        (
            "Signal (V)",
            [
                ("Low-passed", EXAMPLE_PROC["denoised_signal"], {"color": SIGNAL_COLOR}),
                ("Double-exponential fit", EXAMPLE_PROC["fit_signal"], {"color": "k"}),
            ],
        ),
        (
            "Signal Detrended (V)",
            [
                ("Fit subtracted", EXAMPLE_PROC["detrended_signal"], {"color": SIGNAL_COLOR}),
                ("High-pass alternative", highpass["signal"], {"color": "k", "ls": "--"}),
            ],
        ),
        (
            "Control (V)",
            [
                ("Low-passed", EXAMPLE_PROC["denoised_control"], {"color": CONTROL_COLOR}),
                ("Double-exponential fit", EXAMPLE_PROC["fit_control"], {"color": "k"}),
            ],
        ),
        (
            "Control Detrended (V)",
            [
                ("Fit subtracted", EXAMPLE_PROC["detrended_control"], {"color": CONTROL_COLOR}),
                ("High-pass alternative", highpass["control"], {"color": "k", "ls": "--"}),
            ],
        ),
    ],
    EXAMPLE_ZOOM,
    f"A4: Bleaching Correction ({EXAMPLE})",
)
save_figure(fig, "example_A4_bleaching")
plt.show()

#### A5: movement correction

The detrended control is regressed onto the detrended signal (ordinary least squares with an intercept), and
the fitted line is subtracted from the signal. The slope says how much of a control fluctuation appears in the
signal; R² says how much of the signal's variance the control explains. **Look for:** a tight
straight-line scatter if the shared fluctuations are movement-like. A high R² means most of the detrended signal
was shared with the control and has been removed; it does not, by itself, show that what remains is sensor
activity. A second cloud of points off the main line means part of the signal's variance is not shared with the
control (for example a slow dip in the signal alone); the subtraction leaves that part in the corrected trace.

In [ ]:
def plot_regression(ax, control_detrended, signal_detrended, stats, step: int = 20) -> None:
    """Scatter (every *step*-th sample) of detrended signal against detrended control with the fitted line."""
    ax.plot(control_detrended[::step], signal_detrended[::step], ".", ms=2, alpha=0.3, color="0.4")
    x = np.array([control_detrended.min(), control_detrended.max()])
    ax.plot(
        x,
        stats["intercept"] + stats["slope"] * x,
        color="tab:red",
        label=f"slope {stats['slope']:.2f}, R² {stats['r2']:.2f}",
    )
    ax.set(
        xlabel="Detrended Control (V)",
        ylabel="Detrended Signal (V)",
        title="Control-Signal Regression",
    )
    ax.legend()


fig = stage_figure(
    EXAMPLE_TIME,
    [
        (
            "Signal (V)",
            [
                (
                    "Detrended",
                    EXAMPLE_PROC["detrended_signal"],
                    {"color": SIGNAL_COLOR, "alpha": 0.6},
                ),
                ("Estimated movement", EXAMPLE_PROC["movement"], {"color": "tab:orange"}),
            ],
        ),
        ("Corrected (V)", [("Signal minus movement", EXAMPLE_PROC["corrected"], {"color": "k"})]),
    ],
    EXAMPLE_ZOOM,
    f"A5: Movement Correction ({EXAMPLE})",
)
save_figure(fig, "example_A5_movement_time")
plt.show()

fig, ax = plt.subplots(figsize=(4.5, 4), constrained_layout=True)
plot_regression(
    ax,
    EXAMPLE_PROC["detrended_control"],
    EXAMPLE_PROC["detrended_signal"],
    EXAMPLE_PROC["motion_stats"],
)
save_figure(fig, "example_A5_movement_scatter")
plt.show()

#### A6: normalisation

ΔF/F (%) divides the corrected signal by the signal's bleaching fit, so it is a percent change from the
time-varying baseline fluorescence. The session z-score subtracts the session mean and divides by the session SD
of the corrected signal, so it is in units of that session's variability. They have the same shape here
(the baseline changes slowly) and differ in units: z-scores are comparable across sessions only if sessions
have similar variability. Green and red ticks in the zoom mark rewarded and unrewarded outcomes.

In [ ]:
fig = stage_figure(
    EXAMPLE_TIME,
    [
        ("Corrected (V)", [("Signal minus movement", EXAMPLE_PROC["corrected"], {"color": "k"})]),
        ("ΔF/F (%)", [("Session ΔF/F", EXAMPLE_PROC["dff"], {"color": SIGNAL_COLOR})]),
        ("Session z-score", [("Session z-score", EXAMPLE_PROC["zscore"], {"color": "tab:blue"})]),
    ],
    EXAMPLE_ZOOM,
    f"A6: Normalisation ({EXAMPLE})",
    outcome_times=EXAMPLE_OUTCOMES,
)
save_figure(fig, "example_A6_normalisation")
plt.show()

#### Units check: is ΔF/F (%) the right size?

The responses in sections 7-8 are of the order of 0.05-0.1 % ΔF/F, which is small for a norepinephrine sensor. Before reading anything into that,
the arithmetic is checked on the example recording: ΔF/F (%) = 100 x corrected signal (V) / bleaching fit of the signal (V). The fit is the *total*
baseline fluorescence after LED-off subtraction (about 2.6 V here), so 1 % ΔF/F is about 26 mV. The check below prints the
numbers and asserts that ΔF/F equals 100 x corrected / fit sample by sample, and that its SD equals 100 x the SD of the corrected signal over the median fit
(to within the slow variation of the fit).

The table after it gives, **for every recording**, the SD of the session ΔF/F (%) *before* and *after* movement correction, so the size of what
the regression removes is visible. Both are of the low-passed traces used downstream.

In [ ]:
def units_check(processed: dict, raw_signal: np.ndarray) -> dict:
    """Numbers behind the ΔF/F scale of one recording; asserts the x100 and the denominator."""
    fit, corrected, dff = processed["fit_signal"], processed["corrected"], processed["dff"]
    expected_sd_pct = 100 * corrected.std() / np.median(fit)
    assert np.allclose(dff, 100 * corrected / fit)
    assert np.isclose(dff.std(), expected_sd_pct, rtol=0.1), (dff.std(), expected_sd_pct)
    return {
        "median raw signal, LED on minus off (V)": np.median(raw_signal),
        "median bleaching fit F (V)": np.median(fit),
        "SD of corrected signal (V)": corrected.std(),
        "SD of corrected signal as ΔF/F (%)": dff.std(),
        "expected: 100 x SD (V) / median F (%)": expected_sd_pct,
        "1 % ΔF/F corresponds to (mV)": 10 * np.median(fit),
    }


for key, value in units_check(EXAMPLE_PROC, EXAMPLE_REC.signal).items():
    print(f"{key:42s} {value:10.4f}")


def correction_size_row(processed: dict) -> dict:
    """SD of the session ΔF/F (%) before and after movement correction, and the share of the SD removed."""
    before, after = processed["uncorrected_dff"].std(), processed["dff"].std()
    return {
        "sd_dff_before_pct": before,
        "sd_dff_after_pct": after,
        "sd_removed_pct": 100 * (1 - after / before),
        "median_F_v": np.median(processed["fit_signal"]),
    }


correction_size = pd.DataFrame(
    [
        {"recording": rid, "mouse": RECORDINGS[rid].subject, **correction_size_row(p["filtered"])}
        for rid, p in PROCESSED.items()
    ]
)
show_table(correction_size, "qc_correction_size")

## 4. Pipeline B: the nLightG2 paper

Rohner, Curreli, Lamothe-Molina, ..., Patriarchi (2026), "Next-generation multicolor indicators for in vivo imaging of
norepinephrine", *Nature Methods* 23:636-652, <https://doi.org/10.1038/s41592-026-03006-z>.

> **Please check this section against the paper.** The full Methods text could not be retrieved while writing this
> notebook (PubMed Central blocks automated access and the publisher page does not include the Methods). The
> procedure below is implemented from the **methods excerpt that was adapted in cells 40-41 of the earlier
> `photometry_preprocessing` notebook**, not from the paper itself. Anything the excerpt did not specify keeps the
> choice of pipeline A (zero-phase filtering, the same fit bounds). Every number marked "paper" below is as given
> in that adaptation and should be confirmed.

The procedure, as implemented:

| Step | What it does | Setting |
|---|---|---|
| B1 | Low-pass **both** channels (signal and control), Butterworth order 2 | 10 Hz (same function as A3) |
| B2 | Double-exponential bleaching fit and subtraction | same as A4 |
| B3 | Linear regression of the control onto the signal, subtracted | same as A5 |
| B4 | ΔF/F0 (%) = 100 x corrected / bleaching fit | same as A6 |
| B5 | **Event level**: subtract each trial's mean ΔF/F in a pre-event window | −5 to 0 s here (paper: 20 s before tail-lift) |
| B6 | **Trial z-score**: from the *unsmoothed* ΔF/F, subtract the trial's own pre-event mean and divide by its own pre-event SD (sample SD, as in MATLAB) | −5 to 0 s here (paper: −15 to 0 s) |
| B7 | 80-point centred moving mean of each normalised trial; MATLAB `movmean` handling of the even window and of the ends | 80 points = 0.615 s at 130 Hz |
| B8 | Mean ± SEM across trials; peak and area under the curve (AUC) of the mean z-score over a post-event window | 0 to 10 s |

Steps B1-B4 are the *same functions* as A3-A6, so up to ΔF/F the two pipelines produce identical traces. The pipelines differ from B5 on. In the earlier notebooks the −5 to 0 s baselines
were Meg's choice; the paper's 20 s and 15 s windows are configurable (`B_DFF_BASELINE`, `B_Z_BASELINE`) and
widen the extracted windows automatically.

**Event-level normalisation needs a pre-event window of its own, so trials whose baseline contains another
outcome are kept** (the next trial can start within seconds); this is flagged below rather than excluded.

In [ ]:
def interval_mask(rel_time: np.ndarray, interval: tuple[float, float]) -> np.ndarray:
    """Samples with ``start <= t < stop``; the end is excluded so adjacent windows do not share a sample."""
    mask = (rel_time >= interval[0]) & (rel_time < interval[1])
    if mask.sum() < 2:
        raise ValueError(f"interval {interval} holds fewer than two samples of the window.")
    return mask


def matlab_moving_mean(values: np.ndarray, points: int) -> np.ndarray:
    """Centred moving mean along the last axis, as MATLAB's ``movmean``.

    For an even window the samples are ``points/2`` before and ``points/2 - 1`` after the current one;
    near the ends the window shrinks rather than being padded.
    """
    if points < 1 or int(points) != points:
        raise ValueError("the moving-mean window must be a positive integer.")
    values = np.asarray(values, dtype=float)
    n = values.shape[-1]
    index = np.arange(n)
    starts = np.maximum(0, index - points // 2)
    stops = np.minimum(n, index + (points - 1) // 2 + 1)
    sums = np.concatenate([np.zeros(values.shape[:-1] + (1,)), np.cumsum(values, axis=-1)], axis=-1)
    return (sums[..., stops] - sums[..., starts]) / (stops - starts)


def subtract_baseline(
    trials: np.ndarray, rel_time: np.ndarray, baseline: tuple[float, float]
) -> np.ndarray:
    """Step B5: each trial minus its own mean over *baseline*."""
    mask = interval_mask(rel_time, baseline)
    return trials - trials[:, mask].mean(axis=1, keepdims=True)


def trial_zscore(trials: np.ndarray, rel_time: np.ndarray, baseline: tuple[float, float]):
    """Step B6: each trial minus its own baseline mean, divided by its own baseline sample SD.

    Returns:
        tuple[np.ndarray, np.ndarray]: The z-scored trials, and a boolean mask that is False for trials
        whose baseline SD is zero or not finite (those rows are NaN).
    """
    mask = interval_mask(rel_time, baseline)
    mean = trials[:, mask].mean(axis=1, keepdims=True)
    sd = trials[:, mask].std(axis=1, ddof=1, keepdims=True)
    valid = np.isfinite(sd[:, 0]) & (sd[:, 0] > 0)
    with np.errstate(divide="ignore", invalid="ignore"):
        z = np.where(valid[:, None], (trials - mean) / sd, np.nan)
    return z, valid


def normalise_trials_b(trials: np.ndarray, rel_time: np.ndarray, settings: dict):
    """Steps B5-B7 for the ΔF/F windows of one recording.

    Args:
        trials (np.ndarray): ``(n_trials, n_time)`` ΔF/F (%), unsmoothed.
        rel_time (np.ndarray): Time relative to the outcome (s).
        settings (dict): ``dff_baseline``, ``z_baseline`` (s intervals) and ``smoothing_points``.

    Returns:
        tuple[dict, np.ndarray]: ``{"dff": baseline-subtracted and smoothed, "z": trial z-score and smoothed}``
        for the valid trials only, and the validity mask over the input trials.
    """
    z, valid = trial_zscore(trials, rel_time, settings["z_baseline"])
    dff = subtract_baseline(trials, rel_time, settings["dff_baseline"])
    points = settings["smoothing_points"]
    return {
        "dff": matlab_moving_mean(dff[valid], points),
        "z": matlab_moving_mean(z[valid], points),
    }, valid


def mean_and_sem(trials: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Mean over trials and the standard error of the mean (NaN with a single trial)."""
    n = len(trials)
    sem = trials.std(axis=0, ddof=1) / np.sqrt(n) if n > 1 else np.full(trials.shape[1], np.nan)
    return trials.mean(axis=0), sem


def peak_and_auc(rel_time: np.ndarray, trace: np.ndarray, window: tuple[float, float]) -> dict:
    """Largest value, its time, and the signed trapezoidal area of *trace* over *window* (ends included)."""
    mask = (rel_time >= window[0]) & (rel_time <= window[1])
    if mask.sum() < 2:
        raise ValueError(f"window {window} holds fewer than two samples.")
    segment, times = trace[mask], rel_time[mask]
    peak = int(np.argmax(segment))
    return {
        "peak": float(segment[peak]),
        "peak_time_s": float(times[peak]),
        "auc": float(trapezoid(segment, x=times)),
    }


B_SETTINGS = {
    "dff_baseline": B_DFF_BASELINE,
    "z_baseline": B_Z_BASELINE,
    "smoothing_points": B_SMOOTHING_POINTS,
}

### Synthetic check of pipeline B

Trials with a known offset and spread in their pre-event window: after B5 the baseline mean must be zero; after B6
the baseline mean must be zero and the sample SD one; smoothing must give MATLAB's numbers; and a known triangular
response must give the right peak and area.

In [ ]:
def check_pipeline_b_on_synthetic_data() -> pd.DataFrame:
    """Assert the event-level normalisation, smoothing and metrics against hand-computed answers."""
    rng = np.random.default_rng(2)
    rel = relative_time_grid((-5.0, 10.0), 130.0)
    offsets = rng.uniform(-3, 3, size=(40, 1))
    spreads = rng.uniform(0.5, 2.0, size=(40, 1))
    trials = offsets + spreads * rng.standard_normal((40, len(rel)))
    settings = {"dff_baseline": (-5.0, 0.0), "z_baseline": (-5.0, 0.0), "smoothing_points": 1}
    out, valid = normalise_trials_b(trials, rel, settings)  # 1 point = no smoothing
    base = interval_mask(rel, (-5.0, 0.0))
    assert valid.all()
    np.testing.assert_allclose(out["dff"][:, base].mean(axis=1), 0, atol=1e-12)
    np.testing.assert_allclose(out["z"][:, base].mean(axis=1), 0, atol=1e-12)
    np.testing.assert_allclose(out["z"][:, base].std(axis=1, ddof=1), 1, atol=1e-12)
    # MATLAB: movmean([1 2 3 4 5], 4) = [1.5 2 2.5 3.5 4]; movmean(1:5, 3) = [1.5 2 3 4 4.5]
    np.testing.assert_allclose(matlab_moving_mean(np.arange(1.0, 6), 4), [1.5, 2, 2.5, 3.5, 4])
    np.testing.assert_allclose(matlab_moving_mean(np.arange(1.0, 6), 3), [1.5, 2, 3, 4, 4.5])
    # smoothing and averaging over trials commute
    np.testing.assert_allclose(
        matlab_moving_mean(trials, 80).mean(axis=0), matlab_moving_mean(trials.mean(axis=0), 80)
    )
    # a zero-variance baseline is reported invalid instead of dividing by zero
    flat = np.zeros((1, len(rel)))
    assert not trial_zscore(flat, rel, (-5.0, 0.0))[1][0]
    # triangle: 0 at t=0 rising to 2 at t=1 and back to 0 at t=2: peak 2 at 1 s, area 2 z*s
    triangle = np.clip(1 - np.abs(rel - 1), 0, None) * 2
    metrics = peak_and_auc(rel, triangle, (0.0, 10.0))
    assert np.isclose(metrics["peak"], 2) and np.isclose(metrics["peak_time_s"], 1)
    assert np.isclose(metrics["auc"], 2, atol=0.02)
    return pd.DataFrame(
        {
            "check": ["B5/B6 baselines, MATLAB movmean, commutation, invalid trials, peak and AUC"],
            "result": ["passed"],
        }
    )


display(check_pipeline_b_on_synthetic_data())

### Pipeline B on the example recording

Steps B1-B4 are the traces already shown in section 3 (ΔF/F in the A6 figure). The figure below shows the
event-level steps B5-B8 on the example recording's **rewarded** trials: the left column follows six
trials chosen evenly through the session, the right column is the mean ± SEM over all rewarded trials of the
recording at the same step. **Look for:** B5 puts every trial's pre-event (shaded) level at zero; B6 puts it at zero
*and* gives it unit spread, so noisy and quiet trials are on one scale; B7 smooths, and because the moving mean is
centred, the smoothed response begins rising about 0.3 s *before* zero (the 80-point window spans 0.615 s).

In [ ]:
def condition_windows(recording, events: pd.DataFrame, trace: np.ndarray, condition: str, window):
    """Windows of *trace* around the outcomes of *condition* with complete coverage."""
    times = events.loc[events["condition"] == condition, "outcome_time"].to_numpy()
    rel_time, trials, keep = extract_event_windows(
        recording.time, trace, times, window, recording.fs
    )
    return rel_time, trials, keep


def plot_trial_lines(ax, rel_time, trials, shade=None, ylabel="") -> None:
    """Thin lines for a few trials, with an optional shaded interval (the baseline)."""
    for row, colour in zip(trials, plt.cm.viridis(np.linspace(0.1, 0.9, len(trials))), strict=True):
        ax.plot(rel_time, row, color=colour, lw=0.8)
    if shade:
        ax.axvspan(*shade, color="0.5", alpha=0.15)
    ax.axvline(0, color="k", ls="--", lw=0.8)
    ax.set_ylabel(ylabel)


def plot_mean_sem(ax, rel_time, trials, colour="tab:green", shade=None, ylabel="") -> None:
    """Mean ± SEM over trials with the zero line."""
    mean, sem = mean_and_sem(trials)
    ax.plot(rel_time, mean, color=colour, lw=1.4, label=f"n = {len(trials)}")
    ax.fill_between(rel_time, mean - sem, mean + sem, color=colour, alpha=0.25, lw=0)
    if shade:
        ax.axvspan(*shade, color="0.5", alpha=0.15)
    ax.axvline(0, color="k", ls="--", lw=0.8)
    ax.set_ylabel(ylabel)
    ax.legend(loc="upper right", frameon=False)


rel_ex, dff_ex, keep_ex = condition_windows(
    EXAMPLE_REC, EVENTS[EXAMPLE], EXAMPLE_PROC["dff"], "rewarded", EXTRACTION_WINDOW
)
no_smoothing = {**B_SETTINGS, "smoothing_points": 1}
unsmoothed, valid_ex = normalise_trials_b(dff_ex, rel_ex, no_smoothing)
smoothed, _ = normalise_trials_b(dff_ex, rel_ex, B_SETTINGS)
pick = np.linspace(0, valid_ex.sum() - 1, 6).round().astype(int)
steps = [
    ("B4: ΔF/F", "ΔF/F (%)", dff_ex[valid_ex], B_DFF_BASELINE),
    ("B5: Baseline-Subtracted ΔF/F", "Change in ΔF/F (%)", unsmoothed["dff"], B_DFF_BASELINE),
    ("B6: Trial Z-Score, Unsmoothed", "Z-Score (Pre-Event SD)", unsmoothed["z"], B_Z_BASELINE),
    ("B7: Trial Z-Score, 80-Point Mean", "Z-Score (Pre-Event SD)", smoothed["z"], B_Z_BASELINE),
]
fig, axes = plt.subplots(len(steps), 2, figsize=(12, 10), sharex=True, constrained_layout=True)
for (label, ylabel, trials, shade), (ax_trials, ax_mean) in zip(steps, axes, strict=True):
    plot_trial_lines(ax_trials, rel_ex, trials[pick], shade, ylabel)
    plot_mean_sem(ax_mean, rel_ex, trials, "tab:green", shade, ylabel)
    ax_trials.set_title(f"{label}: Six Example Trials", loc="left", fontsize=9)
    ax_mean.set_title(f"{label}: Mean ± SEM, All Rewarded", loc="left", fontsize=9)
for ax in axes[-1]:
    ax.set_xlabel("Time From Reward Onset (s)")
    ax.set_xlim(EVENT_WINDOW)
fig.suptitle(f"Pipeline B Event-Level Steps ({EXAMPLE})", fontsize=12)
save_figure(fig, "example_B_event_level_steps")
plt.show()

## 5. Written comparison of the two pipelines

**What they share.** Both low-pass the signal at 10 Hz with a zero-phase order-2 Butterworth filter, fit a double
exponential to each channel and subtract it, regress the control onto the signal and subtract the fit, and divide by
the bleaching fit to get ΔF/F (%). In this notebook these four steps are literally the same functions
(`lowpass_zero_phase`, `fit_bleaching`, `regress_out_control`, `delta_f_over_f`), so the two pipelines produce
identical ΔF/F traces. The primer also recommends this order (filter, bleaching, movement, normalisation). That the paper's
own steps match is as reported in the methods excerpt only; the paper's text was not available (section 4).

**Where they differ.** Everything after ΔF/F:

| | Pipeline A (primer) | Pipeline B (paper, from the adapted excerpt) |
|---|---|---|
| Normalisation reference | whole session | each trial's own pre-event window |
| ΔF/F offset | none; the level before an event is whatever it was | trial mean of the pre-event window subtracted (default −5 to 0 s; paper: 20 s) |
| z-score | session mean and **session SD** of the corrected signal (`session_zscore`) | trial's pre-event mean and **pre-event SD** of the unsmoothed ΔF/F (default −5 to 0 s; paper: −15 to 0 s) |
| Smoothing | none beyond the 10 Hz low-pass | 80-point centred moving mean after normalisation (0.615 s) |
| Summary | mean ± SEM of the trace | mean ± SEM, plus peak and area under the curve over a post-event window |
| Needs events? | no: the whole trace is normalised once | yes: normalisation is part of the event analysis |

**Consequences, each tied to the code here.**

- *Trial z-scores inflate when the pre-event SD is small.* `trial_zscore` divides by that trial's own pre-event SD, so a
  trial with a quiet baseline gets a large z for the same ΔF/F change; with a 5-s window (650 samples) the SD estimate
  is itself noisy from trial to trial. The session z-score uses one SD for all trials. The table below measures
  how far apart the two denominators are in these recordings.
- *Session z keeps differences between trials in their pre-event level.* Pipeline A does not subtract a baseline, so a
  trial that begins above the session mean stays above it; this slow variability adds to the spread of the mean across
  trials. Pipeline B removes it by construction (and with it any slow, real change that happens to cross the baseline
  window). The table also gives the size of this spread in session-SD units.
- *ΔF/F denominators include autofluorescence.* `delta_f_over_f` divides by the fitted *total* fluorescence of the signal
  channel after LED-off subtraction, which contains the fibre's and tissue's autofluorescence as well as the
  indicator. The relative change of the indicator alone would be larger. This is true of both pipelines, so ΔF/F
  values are comparable between them but not directly with literature values from other set-ups.
- *The 80-point smoothing spreads a response in time.* The moving mean is centred, so each smoothed sample averages
  points up to 0.3 s ahead: a step at zero begins to rise about 0.3 s *before* zero (computed below). It also lowers
  peaks and slows apparent onsets. Pipeline A's traces are not affected beyond the 10 Hz low-pass.
- *Peak and area are defined on different scales.* Pipeline B's peak and AUC are in trial pre-event-SD units of smoothed
  traces; pipeline A's are in session-SD units. They are not interchangeable.

Not verified for the nLightG2 paper: its exact baseline windows, whether it z-scores per trial or per animal, its
smoothing length, and how it reports peak and AUC; all of these are taken from the adapted excerpt.

In [ ]:
def baseline_comparison(recording, events, dff, window, baseline) -> dict:
    """Session SD against pre-event SD and between-trial baseline spread for one recording's ΔF/F."""
    _, trials, _ = extract_event_windows(
        recording.time, dff, events["outcome_time"].to_numpy(), window, recording.fs
    )
    rel_time = relative_time_grid(window, recording.fs)
    mask = interval_mask(rel_time, baseline)
    pre_sd = trials[:, mask].std(axis=1, ddof=1)
    pre_mean = trials[:, mask].mean(axis=1)
    session_sd = dff.std()
    return {
        "n_trials": len(trials),
        "session_sd_pct": session_sd,
        "median_pre_event_sd_pct": np.median(pre_sd),
        "median_session_sd_over_pre_sd": np.median(session_sd / pre_sd),
        "pre_sd_below_half_session_sd": np.mean(pre_sd < 0.5 * session_sd),
        "trial_to_trial_sd_of_pre_mean_over_session_sd": (
            pre_mean.std(ddof=1) / session_sd if len(pre_mean) > 1 else np.nan
        ),
    }


rows = []
for rid, rec in RECORDINGS.items():
    stats = baseline_comparison(
        rec, EVENTS[rid], PROCESSED[rid]["filtered"]["dff"], EXTRACTION_WINDOW, B_Z_BASELINE
    )
    rows.append({"recording": rid, **stats})
baseline_spread = pd.DataFrame(rows)
show_table(baseline_spread, "pipeline_comparison_baselines")

step = np.where(relative_time_grid((-5.0, 5.0), 130.0) >= 0, 1.0, 0.0)
smoothed_step = matlab_moving_mean(step, B_SMOOTHING_POINTS)
grid = relative_time_grid((-5.0, 5.0), 130.0)
onset = grid[np.argmax(smoothed_step > 0.01)]
print(
    f"A unit step at 0 s, smoothed with {B_SMOOTHING_POINTS} points at 130 Hz, first exceeds 1% of its height at "
    f"{onset:.2f} s and reaches half height at {grid[np.argmax(smoothed_step >= 0.5)]:.2f} s."
)

## 6. Quality control suite

Each QC function takes the recordings (or their processed traces) and returns a **tidy table with one row per
recording**. Every table is printed and saved as CSV, and carries a `flags` column listing which thresholds
(`QC_THRESHOLDS` in the configuration cell) the recording crosses. **Flags are for you to look at; nothing is
excluded.**

The tables follow the processing order:

| Table | Step | Question |
|---|---|---|
| 6.1 acquisition | A1 and sync | Was the recording healthy: light levels, clipping, dropouts, sync coverage? |
| 6.2 denoising | A3 | How much high-frequency noise was there and how much did the low-pass remove? |
| 6.3 bleaching | A4 | How much did each channel decay, and is the fit sensible? |
| 6.4 movement | A5 | How much of the signal was shared with the control, and are there connection jumps? |
| 6.5 SNR | final traces | How far do slow fluctuations sit above the noise floor, and is it specific to the sensor? |
| 8b (after section 8) | final traces | How large is the evoked response relative to random times? Per-mouse ranking. |

Pipelines A and B share steps A1-A5 (section 4), so tables 6.1-6.4 and the spectral SNR describe both. Evoked SNR depends
on how events are normalised, so it is computed for each pipeline's final traces; it needs the outcome-aligned windows and the
random-time control of sections 7-8 and is therefore finished in **section 8b**, which also holds the per-mouse
ranking.

### How the SNR measures are defined

- **Spectral SNR (primary, event-independent).** The movement-corrected ΔF/F *before* the low-pass is split into the
  power spectral density (Welch, 30-s segments). SNR (dB) = 10 log10 of the mean PSD at 0.1-3 Hz (where the sensor
  fluctuations of interest live) over the mean PSD at 15-50 Hz (a noise floor, where nothing physiological is expected). It says how far the
  slow fluctuations sit above the noise floor. The same number for the **control** channel (its own ΔF/F) gives
  `specificity_db` = **uncorrected sensor SNR minus control SNR**, both ΔF/F of their own bleaching fit and *before* the movement regression
  (comparing the corrected sensor with the raw control would penalise the sensor for the correction). About **0 dB means the slow
  fluctuations are equally present in the control channel**, i.e. whatever drives the sensor's slow variance also drives the
  control (movement, haemodynamics, or a control that is not isosbestic); positive means the sensor carries slow power the control
  lacks, which is what a sensor responding to norepinephrine should show.
- **Robust noise floor** `sigma_noise_pct`: 1.4826 x the median absolute deviation of the >15 Hz component of that
  ΔF/F (percent). It is the noise outside the signal band, robust to the few large events. **Transient SNR:** transients are
  peaks of the 10 Hz low-passed ΔF/F whose prominence exceeds k = 5 robust SDs of that trace; we report them per minute and
  their median prominence divided by `sigma_noise_pct`.
- **Evoked SNR (section 8b).** For each rewarded trial, response = mean z over 0-2 s minus mean z over −2-0 s. The
  mean response is summarised by d′ = mean / SD of the response across trials (an effect size that does not depend on the number of trials;
  **used for the ranking**) and, for significance only, by its z against the same measure at the 200 random control times (z grows with √n trials).

In [ ]:
def run_lengths(mask: np.ndarray) -> np.ndarray:
    """Lengths of the runs of consecutive True values in *mask*."""
    padded = np.concatenate([[False], np.asarray(mask, dtype=bool), [False]])
    edges = np.flatnonzero(padded[1:] != padded[:-1])
    return edges[1::2] - edges[::2]


def robust_sd(x: np.ndarray) -> float:
    """1.4826 x median absolute deviation: the standard deviation of a normal distribution, robust to outliers."""
    return float(1.4826 * np.median(np.abs(x - np.median(x))))


def attach_flags(table: pd.DataFrame, rules: dict[str, pd.Series]) -> pd.DataFrame:
    """Add a ``flags`` column listing the names of the rules each row breaks (comparisons with NaN are False)."""
    broken = pd.DataFrame(rules, index=table.index).fillna(False).astype(bool)
    out = table.copy()
    out["flags"] = broken.apply(lambda row: ", ".join(row.index[row.to_numpy()]), axis=1)
    return out


def per_recording(
    recordings: dict, row_function: Callable, processed: dict | None = None, **settings
) -> pd.DataFrame:
    """One row per recording: ``row_function(rec, processed[rid], **settings)`` returns a dict of columns.

    *processed* (optional) maps recording id to that recording's processed traces; without it the
    row function is called as ``row_function(rec, **settings)``. Nothing is read from the notebook's globals.
    """
    rows = []
    for rid, rec in recordings.items():
        extra = () if processed is None else (processed[rid],)
        rows.append(
            {
                "recording": rid,
                "mouse": rec.subject,
                "session": rec.session,
                **row_function(rec, *extra, **settings),
            }
        )
    return pd.DataFrame(rows)

### 6.1 Acquisition (step A1 and sync)

Light levels (median and 1st/99th percentile of the baseline-subtracted voltage per channel), the LED-off baseline level,
the fraction of samples where the ADC clipped, runs of identical samples longer than 0.5 s (a frozen
recording), the sync match and the fraction of samples outside the sync coverage. The LED-off baseline can contain brief
upward spikes (counted as "baseline spikes per minute": runs above the baseline median plus `QC_BASELINE_SPIKE_SD` = 6 robust SDs), which subtract as
small downward blips from the LED-on signal. **Look for:** no clipping, no flat runs, baselines of a few tens of mV,
`fraction_matched` near 1, and little of the recording outside the sync coverage.

In [ ]:
def baseline_spikes_per_min(baseline: np.ndarray, fs: float, spike_sd: float) -> float:
    """Brief upward excursions of the LED-off baseline per minute (runs above median + *spike_sd* robust SD)."""
    above = baseline > np.median(baseline) + spike_sd * robust_sd(baseline)
    return len(run_lengths(above)) / (len(baseline) / fs / 60)


def acquisition_row(rec, flatline_s: float, spike_sd: float) -> dict:
    """The acquisition QC columns of one recording; runs of identical samples over *flatline_s* count as dropouts."""
    flat = [
        run_lengths(np.diff(x) == 0) + 1  # n identical consecutive diffs = n + 1 identical samples
        for x in (rec.signal_led_on, rec.control_led_on)
    ]
    flat_s = np.concatenate(flat) / rec.fs
    quality = rec.sync_quality
    row = {
        "duration_min": len(rec.signal) / rec.fs / 60,
        "led_current_ma_1": rec.led_current[0],
        "led_current_ma_2": rec.led_current[1],
    }
    for name, channel, baseline, clipping in (
        ("signal", rec.signal, rec.signal_baseline, rec.signal_clipping),
        ("control", rec.control, rec.control_baseline, rec.control_clipping),
    ):
        p01, median, p99 = np.percentile(channel, [1, 50, 99])
        row |= {
            f"{name}_median_v": median,
            f"{name}_p01_v": p01,
            f"{name}_p99_v": p99,
            f"{name}_baseline_v": np.median(baseline),
            f"{name}_baseline_spikes_per_min": baseline_spikes_per_min(baseline, rec.fs, spike_sd),
            f"{name}_clipping_fraction": clipping.mean(),
        }
    return row | {
        "flatline_runs": int((flat_s > flatline_s).sum()),
        "longest_flat_s": float(flat_s.max()),
        "sync_fraction_matched": quality["fraction_matched"],
        "sync_max_discrepancy_samples": quality["max_abs_interval_discrepancy_ms"] * rec.fs / 1000,
        "fraction_outside_sync": rec.n_samples_outside_sync / len(rec.signal),
    }


def acquisition_qc(
    recordings: dict, thresholds: dict, flatline_s: float, spike_sd: float
) -> pd.DataFrame:
    """Table 6.1 for every recording, with flags."""
    table = per_recording(recordings, acquisition_row, flatline_s=flatline_s, spike_sd=spike_sd)
    clipping = table[["signal_clipping_fraction", "control_clipping_fraction"]].max(axis=1)
    baseline = table[["signal_baseline_v", "control_baseline_v"]].max(axis=1)
    return attach_flags(
        table,
        {
            "short": table["duration_min"] < thresholds["min_duration_min"],
            "high_baseline": baseline > thresholds["max_baseline_v"],
            "clipping": clipping > thresholds["max_clipping_fraction"],
            "flatline": table["flatline_runs"] > thresholds["max_flatline_runs"],
            "sync_fraction": table["sync_fraction_matched"] < thresholds["min_sync_fraction"],
            "sync_discrepancy": table["sync_max_discrepancy_samples"]
            > thresholds["max_sync_discrepancy_samples"],
            "outside_sync": table["fraction_outside_sync"]
            > thresholds["max_outside_sync_fraction"],
        },
    )

### 6.2 Denoising (step A3)

High-frequency noise is the standard deviation of the >15 Hz component (a zero-phase high-pass at 15 Hz), measured before and
after the low-pass, in millivolts and as a percentage of the median signal level. **Look for:** `noise_after` far below
`noise_before` (the filter is doing its job); a `noise_before_pct` of a fraction of a percent. The noise is white-ish, so a recording with
much more noise than the others has a poorer light path or detector gain.

In [ ]:
def hf_noise_sd(x: np.ndarray, fs: float, cutoff_hz: float) -> float:
    """Standard deviation of the part of *x* above *cutoff_hz*."""
    return float(highpass_zero_phase(x, fs, cutoff_hz).std())


def denoising_row(rec, processed_traces: dict, cutoff_hz: float) -> dict:
    """Denoising QC columns of one recording; *processed_traces* is its ``PROCESSED`` entry."""
    processed = processed_traces["filtered"]
    row = {}
    for name, raw in (("signal", rec.signal), ("control", rec.control)):
        before = hf_noise_sd(raw, rec.fs, cutoff_hz)
        after = hf_noise_sd(processed[f"denoised_{name}"], rec.fs, cutoff_hz)
        row |= {
            f"{name}_noise_before_mv": 1000 * before,
            f"{name}_noise_after_mv": 1000 * after,
            f"{name}_noise_removed_pct": 100 * (1 - after / before),
            f"{name}_noise_before_pct": 100 * before / np.median(raw),
        }
    return row


def denoising_qc(
    recordings: dict, processed: dict, thresholds: dict, cutoff_hz: float
) -> pd.DataFrame:
    """Table 6.2 for every recording, with flags."""
    table = per_recording(recordings, denoising_row, processed, cutoff_hz=cutoff_hz)
    relative = table[["signal_noise_before_pct", "control_noise_before_pct"]].max(axis=1)
    return attach_flags(table, {"noisy": relative > thresholds["max_hf_noise_pct"]})

### 6.3 Bleaching (step A4)

The drop of each fitted curve from the first to the last sample (%), the root-mean-square error between the low-passed
channel and its fit (mV; this contains all real fluctuations, so it is a scale, not a goodness-of-fit test), and which of
the five fit parameters sit on a bound (within 0.1% of the range). A parameter on a bound is common in a
20-minute recording, where the decay is too slow to identify its time constants, and it does not make the curve wrong; many
parameters on bounds mean the *parameters* are not interpretable. **Look for:** drops of a few to some tens of percent, and a
control that decays at a similar rate to the signal.

In [ ]:
def params_at_bounds(params: np.ndarray, bounds: tuple, tolerance: float = 1e-3) -> list[str]:
    """Names of the fit parameters within *tolerance* of the range of a bound."""
    lower, upper = bounds
    span = upper - lower
    on_bound = (np.abs(params - lower) <= tolerance * span) | (
        np.abs(params - upper) <= tolerance * span
    )
    return [name for name, hit in zip(PARAMETER_NAMES, on_bound, strict=True) if hit]


def bleaching_row(rec, processed_traces: dict) -> dict:
    """Bleaching QC columns of one recording; *processed_traces* is its ``PROCESSED`` entry."""
    processed = processed_traces["filtered"]
    row = {}
    for name in ("signal", "control"):
        fit = processed[f"fit_{name}"]
        at_bounds = params_at_bounds(processed[f"params_{name}"], processed[f"bounds_{name}"])
        row |= {
            f"{name}_drop_pct": 100 * (fit[0] - fit[-1]) / fit[0],
            f"{name}_fit_rmse_mv": 1000
            * float(np.sqrt(np.mean((processed[f"denoised_{name}"] - fit) ** 2))),
            f"{name}_tau_slow_s": processed[f"params_{name}"][3],
            f"{name}_n_params_at_bound": len(at_bounds),
            f"{name}_params_at_bound": " ".join(at_bounds),
        }
    return row


def bleaching_qc(recordings: dict, processed: dict, thresholds: dict) -> pd.DataFrame:
    """Table 6.3 for every recording, with flags."""
    table = per_recording(recordings, bleaching_row, processed)
    drop = table[["signal_drop_pct", "control_drop_pct"]].abs().max(axis=1)
    at_bound = table[["signal_n_params_at_bound", "control_n_params_at_bound"]].max(axis=1)
    return attach_flags(
        table,
        {
            "large_drop": drop > thresholds["max_bleaching_drop_pct"],
            "many_params_at_bound": at_bound > thresholds["max_params_at_bound"],
        },
    )

### 6.4 Movement (step A5) and connection jumps

The Pearson correlation of the detrended signal with the detrended control, the regression slope, R², and the percentage
of the detrended signal's variance the subtraction removed (for a least-squares fit with an intercept this equals
100 x R², which is a useful consistency check). **Connection jumps**: sudden steps when a fibre connection shifts. They are found as peaks
of a **0.5-s mean-difference step detector** on the control (mean of the next 0.5 s minus mean of the previous 0.5 s) above
10 robust SDs (1.4826 x MAD of the step trace), and kept only if the *signal* shows a fractional change of **the same sign** at that moment (a
real light-path change moves both channels the same way). Because chance alone gives equal signs half of the time, the
rate is an upper bound on true jumps. We report jumps per minute and the median size (% of the fitted baseline).
**Look for:** a slope near 1, a high R² (most of the signal's variance is shared), and a jump rate near zero.

In [ ]:
def mean_difference_step(x: np.ndarray, width: int) -> tuple[np.ndarray, np.ndarray]:
    """Mean of the next *width* samples minus the mean of the previous *width*, at each boundary.

    Returns:
        tuple[np.ndarray, np.ndarray]: Index of the first sample after each boundary, and the step.
    """
    cumulative = np.concatenate([[0.0], np.cumsum(x)])
    boundary = np.arange(width, len(x) - width + 1)
    after = (cumulative[boundary + width] - cumulative[boundary]) / width
    before = (cumulative[boundary] - cumulative[boundary - width]) / width
    return boundary, after - before


def detect_jumps(
    control_detrended, signal_detrended, control_fit, signal_fit, fs, window_s, mad_threshold
) -> pd.DataFrame:
    """Candidate connection jumps on the control, confirmed when the signal changes the same way.

    Returns:
        pd.DataFrame: One row per candidate with ``sample``, ``control_pct`` and ``signal_pct`` (step as % of
        the fitted baseline in each channel) and ``confirmed`` (equal sign).
    """
    width = int(round(window_s * fs))
    boundary, step_control = mean_difference_step(control_detrended, width)
    _, step_signal = mean_difference_step(signal_detrended, width)
    threshold = mad_threshold * robust_sd(step_control)
    peaks, _ = sps.find_peaks(np.abs(step_control), height=threshold, distance=width)
    sample = boundary[peaks]
    control_pct = 100 * step_control[peaks] / control_fit[sample]
    signal_pct = 100 * step_signal[peaks] / signal_fit[sample]
    return pd.DataFrame(
        {
            "sample": sample,
            "control_pct": control_pct,
            "signal_pct": signal_pct,
            "confirmed": np.sign(control_pct) == np.sign(signal_pct),
        }
    )


def movement_row(rec, processed_traces: dict, window_s: float, jump_threshold: float) -> dict:
    """Movement-correction QC columns of one recording; *processed_traces* is its ``PROCESSED`` entry."""
    processed = processed_traces["filtered"]
    stats = processed["motion_stats"]
    removed = 100 * (1 - processed["corrected"].var() / processed["detrended_signal"].var())
    jumps = detect_jumps(
        processed["detrended_control"],
        processed["detrended_signal"],
        processed["fit_control"],
        processed["fit_signal"],
        rec.fs,
        window_s,
        jump_threshold,
    )
    confirmed = jumps[jumps["confirmed"]]
    return {
        "pearson_r": stats["r"],
        "slope": stats["slope"],
        "r2": stats["r2"],
        "variance_removed_pct": removed,
        "jumps_per_min": len(confirmed) / (len(rec.signal) / rec.fs / 60),
        "n_jumps": len(confirmed),
        "median_jump_size_pct": confirmed["control_pct"].abs().median()
        if len(confirmed)
        else np.nan,
    }


def movement_qc(
    recordings: dict, processed: dict, thresholds: dict, window_s: float, jump_threshold: float
) -> pd.DataFrame:
    """Table 6.4 for every recording, with flags."""
    table = per_recording(
        recordings, movement_row, processed, window_s=window_s, jump_threshold=jump_threshold
    )
    return attach_flags(
        table,
        {
            "non_positive_slope": table["slope"] <= thresholds["min_movement_slope"],
            "many_jumps": table["jumps_per_min"] > thresholds["max_jumps_per_min"],
        },
    )

### 6.5 Signal-to-noise of the final traces

Definitions are given at the start of this section. **Look for:** a positive `snr_db` (slow fluctuations above the noise
floor), a positive `specificity_db` (the *uncorrected* sensor beats the control), and
`transient_amp_over_noise` well above the detection threshold of k = 5.

**Reading `specificity_db`** (default flag threshold `min_specificity_db` = 1 dB): the sensor's *uncorrected* SNR minus the control's SNR. A value
near 0 dB means slow fluctuations are equally present in both channels; the movement regression can then only remove what is common, and little
sensor-specific slow signal is expected. `snr_db` (the movement-corrected sensor) remains the primary SNR; `uncorrected_snr_db` is the sensor before
step A5.

In [ ]:
def welch_band_means(x: np.ndarray, fs: float, bands: list[tuple[float, float]], segment_s: float):
    """Mean power spectral density (Welch) of *x* inside each band (Hz)."""
    freqs, psd = sps.welch(x, fs=fs, nperseg=min(int(segment_s * fs), len(x)), detrend="constant")
    return [float(psd[(freqs >= lo) & (freqs <= hi)].mean()) for lo, hi in bands]


def spectral_snr_db(x: np.ndarray, fs: float, signal_band, noise_band, segment_s: float) -> float:
    """10 log10 of mean PSD in *signal_band* over mean PSD in *noise_band*."""
    slow, noise = welch_band_means(x, fs, [signal_band, noise_band], segment_s)
    return 10 * np.log10(slow / noise)


def noise_floor_sigma(x: np.ndarray, fs: float, cutoff_hz: float) -> float:
    """Robust SD of the part of *x* above *cutoff_hz*."""
    return robust_sd(highpass_zero_phase(x, fs, cutoff_hz))


def transient_stats(lowpassed: np.ndarray, fs: float, sigma_noise: float, k: float) -> dict:
    """Transients of a low-passed trace: peaks with prominence above *k* robust SDs of the trace."""
    prominences = sps.find_peaks(lowpassed, prominence=k * robust_sd(lowpassed))[1]["prominences"]
    minutes = len(lowpassed) / fs / 60
    return {
        "transients_per_min": len(prominences) / minutes,
        "transient_amp_over_noise": float(np.median(prominences) / sigma_noise)
        if len(prominences)
        else np.nan,
    }


def snr_row(
    rec, processed_traces: dict, signal_band, noise_band, segment_s: float, prominence_sd: float
) -> dict:
    """Spectral and transient SNR columns of one recording; *processed_traces* is its ``PROCESSED`` entry."""
    unfiltered = processed_traces["unfiltered"]
    bands = (signal_band, noise_band, segment_s)
    sensor_db = spectral_snr_db(unfiltered["dff"], rec.fs, *bands)
    control_db = spectral_snr_db(unfiltered["control_dff"], rec.fs, *bands)
    sigma = noise_floor_sigma(unfiltered["dff"], rec.fs, noise_band[0])
    uncorrected_db = spectral_snr_db(unfiltered["uncorrected_dff"], rec.fs, *bands)
    return {
        "snr_db": sensor_db,
        "control_snr_db": control_db,
        "uncorrected_snr_db": uncorrected_db,
        "specificity_db": uncorrected_db - control_db,
        "sigma_noise_pct": sigma,
        **transient_stats(processed_traces["filtered"]["dff"], rec.fs, sigma, prominence_sd),
    }


def snr_qc(
    recordings: dict,
    processed: dict,
    thresholds: dict,
    signal_band,
    noise_band,
    segment_s: float,
    prominence_sd: float,
) -> pd.DataFrame:
    """Table 6.5 for every recording, with flags."""
    table = per_recording(
        recordings,
        snr_row,
        processed,
        signal_band=signal_band,
        noise_band=noise_band,
        segment_s=segment_s,
        prominence_sd=prominence_sd,
    )
    return attach_flags(
        table,
        {
            "low_snr": table["snr_db"] < thresholds["min_snr_db"],
            "low_specificity": table["specificity_db"] < thresholds["min_specificity_db"],
        },
    )

### Sanity check of the QC functions on synthetic data

Each measure is run on data with a **known** answer before it is used on real recordings. The spectral SNR must recover the
calculated value (a sum of equal-amplitude sinusoids at 0.1-3 Hz over white noise) to within 1 dB, and fall as the noise
rises; the noise floor must recover the white-noise level within the fraction that the >15 Hz band holds; transient
counts and sizes must match the injected ones; the jump detector must find injected steps and reject one whose sign
disagrees between channels; the evoked measures must recover a known response and d′.

In [ ]:
def synthetic_slow_plus_noise(
    rng, fs: float, minutes: float, noise_sd: float, n_tones: int = 50, tone_amp: float = 1.0
):
    """Equal-amplitude sinusoids at 0.1-3 Hz plus white noise, and the SNR (dB) the Welch ratio should give."""
    n = int(minutes * 60 * fs)
    t = np.arange(n) / fs
    freqs = np.linspace(0.15, 2.95, n_tones)
    slow = sum(tone_amp * np.sin(2 * np.pi * f * t + rng.uniform(0, 2 * np.pi)) for f in freqs)
    slow_psd = n_tones * tone_amp**2 / 2 / (QC_SIGNAL_BAND_HZ[1] - QC_SIGNAL_BAND_HZ[0])
    noise_psd = 2 * noise_sd**2 / fs  # one-sided PSD of white noise
    return slow + noise_sd * rng.standard_normal(n), 10 * np.log10(slow_psd / noise_psd)


def check_qc_functions_on_synthetic_data() -> pd.DataFrame:
    """Assert that each QC measure recovers a known answer; return the comparison."""
    fs, rng = 130.0, np.random.default_rng(3)
    rows = []

    snrs = []
    for noise_sd in (0.5, 1.0, 2.0):
        x, expected = synthetic_slow_plus_noise(rng, fs, 10.0, noise_sd)
        measured = spectral_snr_db(x, fs, QC_SIGNAL_BAND_HZ, QC_NOISE_BAND_HZ, QC_WELCH_SEGMENT_S)
        assert abs(measured - expected) < 1.0, (noise_sd, measured, expected)
        snrs.append(measured)
        rows.append((f"Spectral SNR, noise SD {noise_sd}", measured, expected))
    assert snrs[0] > snrs[1] > snrs[2], snrs

    sigma = noise_floor_sigma(0.5 * rng.standard_normal(int(120 * fs)), fs, QC_NOISE_BAND_HZ[0])
    share = 1 - QC_NOISE_BAND_HZ[0] / (fs / 2)  # fraction of white-noise variance above the cutoff
    assert abs(sigma / (0.5 * np.sqrt(share)) - 1) < 0.1, sigma
    rows.append(
        ("Noise floor (SD of >15 Hz part of white noise SD 0.5)", sigma, 0.5 * np.sqrt(share))
    )

    n_events, amplitude = 20, 5.0
    t = np.arange(int(300 * fs)) / fs
    onsets = np.linspace(10, 280, n_events)
    for noise_sd in (0.3, 0.6):
        trace = amplitude * sum(
            transient_shape(t, onset) for onset in onsets
        ) + noise_sd * rng.standard_normal(len(t))
        low = lowpass_zero_phase(trace, fs, 10.0)
        sigma_noise = noise_floor_sigma(trace, fs, QC_NOISE_BAND_HZ[0])
        stats = transient_stats(low, fs, sigma_noise, QC_TRANSIENT_PROMINENCE_SD)
        assert abs(stats["transients_per_min"] * 5 - n_events) <= 2, stats
        ratio = stats["transient_amp_over_noise"]
        assert abs(ratio / (amplitude / sigma_noise) - 1) < 0.25, (ratio, amplitude / sigma_noise)
        rows.append(
            (f"Transient amplitude / noise, noise SD {noise_sd}", ratio, amplitude / sigma_noise)
        )

    n = 130 * 60
    base_c, base_s = np.full(n, 2.0), np.full(n, 2.5)
    control = 0.004 * rng.standard_normal(n)
    signal = 0.004 * rng.standard_normal(n)
    control, signal = lowpass_zero_phase(control, fs, 10.0), lowpass_zero_phase(signal, fs, 10.0)
    for second, d_control, d_signal in ((20, 0.2, 0.25), (40, -0.1, -0.12), (50, 0.2, -0.25)):
        control[int(second * fs) :] += d_control
        signal[int(second * fs) :] += d_signal
    jumps = detect_jumps(
        control, signal, base_c, base_s, fs, QC_JUMP_WINDOW_S, QC_JUMP_MAD_THRESHOLD
    )
    assert len(jumps) == 3 and jumps["confirmed"].tolist() == [True, True, False], jumps
    np.testing.assert_allclose(jumps["sample"] / fs, [20, 40, 50], atol=0.02)
    np.testing.assert_allclose(jumps["control_pct"], [10, -5, 10], rtol=0.1)
    rows.append(("Jumps found / confirmed (injected 3 / 2)", len(jumps), 3))
    return pd.DataFrame(rows, columns=["check", "measured", "expected"])


display(check_qc_functions_on_synthetic_data())

### QC tables for every recording

Run in order. Each table is saved as CSV; the combined list of flags follows them.

In [ ]:
qc_acquisition = acquisition_qc(RECORDINGS, QC_THRESHOLDS, QC_FLATLINE_S, QC_BASELINE_SPIKE_SD)
show_table(qc_acquisition, "qc_1_acquisition")

In [ ]:
qc_denoising = denoising_qc(RECORDINGS, PROCESSED, QC_THRESHOLDS, QC_NOISE_BAND_HZ[0])
show_table(qc_denoising, "qc_2_denoising")

In [ ]:
qc_bleaching = bleaching_qc(RECORDINGS, PROCESSED, QC_THRESHOLDS)
show_table(qc_bleaching, "qc_3_bleaching")

In [ ]:
qc_movement = movement_qc(
    RECORDINGS, PROCESSED, QC_THRESHOLDS, QC_JUMP_WINDOW_S, QC_JUMP_MAD_THRESHOLD
)
show_table(qc_movement, "qc_4_movement")

In [ ]:
qc_snr = snr_qc(
    RECORDINGS,
    PROCESSED,
    QC_THRESHOLDS,
    QC_SIGNAL_BAND_HZ,
    QC_NOISE_BAND_HZ,
    QC_WELCH_SEGMENT_S,
    QC_TRANSIENT_PROMINENCE_SD,
)
show_table(qc_snr, "qc_5_snr")

In [ ]:
def combined_flags(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """One row per recording with the flags from each QC table."""
    out = next(iter(tables.values()))[["recording", "mouse", "session"]].copy()
    for name, table in tables.items():
        out[name] = table["flags"].to_numpy()
    return out


qc_flags = combined_flags(
    {
        "acquisition": qc_acquisition,
        "denoising": qc_denoising,
        "bleaching": qc_bleaching,
        "movement": qc_movement,
        "snr": qc_snr,
    }
)
show_table(qc_flags, "qc_flags")

## 7. Outcome-aligned responses, per mouse

**Events.** Rewarded trials are aligned to `outcome_time` with kind `reward_onset` (entry to the `choice` state, the onset
of reward pulse 1). Unrewarded trials are aligned to the `no_reward_cue_onset` (entry to `ITI`, the onset of the no-reward
cue). Both come from `load_outcome_events`; trials of the phases in `PHASES` are used.

**Windows.** −5 to +10 s around each event (`EVENT_WINDOW`), interpolated onto a common grid at the recording rate (130 Hz).
Only windows lying completely inside the sync coverage **and clear of any stretch of missing sync time** are kept (a window is never
padded, extrapolated, or interpolated across a gap); the exclusions are listed below, with the trials whose
pre-event baseline contains another outcome (kept, as in the earlier notebooks) and any trial whose baseline SD is zero
(dropped from all four views so both pipelines use the same trials).

**Pooling.** Each recording is normalised on its own (pipeline A: its session ΔF/F and session z-score; pipeline B: each
trial's own baseline), and then the trials of a mouse's recordings are pooled. A mouse with several sessions
therefore contributes more trials, and **the shaded bands are the SEM across trials, not across mice** (the cohort is too
small and the trial counts too unequal for between-mouse statistics here; trials of one session are not independent either).

**Panels.** Rows are mice. Left column: ΔF/F (pipeline A: session ΔF/F; pipeline B: baseline-subtracted and smoothed). Right
column: z-score (A: session z-score; B: trial z-score, smoothed). Rewarded in green, unrewarded in red; `n` is the number
of trials. The dashed vertical line is the event. **What to look for:** a response that follows the event in the same direction
in many mice; rewarded and unrewarded traces that diverge after 0; a flat trace before 0.

In [ ]:
CONDITIONS = ("rewarded", "unrewarded")
CONDITION_COLORS = {"rewarded": "tab:green", "unrewarded": "tab:red"}
VIEWS = ("A_dff", "A_z", "B_dff", "B_z")
VIEW_LABELS = {
    "A_dff": ("Session ΔF/F", "ΔF/F (%)"),
    "A_z": ("Session Z-Score", "Z-Score (Session SD)"),
    "B_dff": ("Baseline-Subtracted ΔF/F, Smoothed", "Change in ΔF/F (%)"),
    "B_z": ("Trial Z-Score, Smoothed", "Z-Score (Pre-Event SD)"),
    "signal_uncorrected": ("(a) Signal ΔF/F, Before Movement Correction", "ΔF/F (%)"),
    "control": ("(b) Control ΔF/F (Its Own Fit)", "ΔF/F (%)"),
    "control_contribution": ("(c) Fitted Control Contribution (Subtracted)", "ΔF/F (%)"),
    "corrected": ("(d) Corrected Signal ΔF/F", "ΔF/F (%)"),
}
CHANNEL_VIEWS = ("signal_uncorrected", "control", "control_contribution", "corrected")


def channel_traces(p: dict) -> dict[str, np.ndarray]:
    """Whole-session traces (ΔF/F %) for the control-channel check: (a) = (c) + (d) exactly.

    *p* is a recording's ``PROCESSED[rid]["filtered"]``. The fitted control contribution is the estimated movement
    component (V) expressed as % of the signal's bleaching fit.
    """
    return {
        "signal_uncorrected": p["uncorrected_dff"],
        "control": p["control_dff"],
        "control_contribution": delta_f_over_f(p["movement"], p["fit_signal"]),
        "corrected": p["dff"],
    }


def compute_views(dff_windows: np.ndarray, z_windows: np.ndarray, rel_time: np.ndarray, b_settings):
    """The four displayed versions of one set of event windows, restricted to the same trials.

    Args:
        dff_windows, z_windows (np.ndarray): ``(n, n_time)`` windows of the session ΔF/F and the session z-score.
        rel_time (np.ndarray): Time relative to the event (s).
        b_settings: Pipeline B's baseline and smoothing settings (``B_SETTINGS``).

    Returns:
        tuple[dict, np.ndarray]: Arrays by view name (``A_dff``, ``A_z``, ``B_dff``, ``B_z``) for the valid trials,
        and the validity mask (False where the pipeline-B trial baseline had zero SD).
    """
    b_views, valid = normalise_trials_b(dff_windows, rel_time, b_settings)
    return {
        "A_dff": dff_windows[valid],
        "A_z": z_windows[valid],
        "B_dff": b_views["dff"],
        "B_z": b_views["z"],
    }, valid


def trace_windows(rec, processed: dict, event_times: np.ndarray, window: tuple[float, float]):
    """ΔF/F and z-score windows of a recording around *event_times* (complete windows only).

    *processed* is the recording's ``PROCESSED[rid]["filtered"]``; *window* is relative to each event (s).
    """
    rel_time, dff_windows, keep = extract_event_windows(
        rec.time, processed["dff"], event_times, window, rec.fs
    )
    _, z_windows, _ = extract_event_windows(
        rec.time, processed["zscore"], event_times, window, rec.fs
    )
    return rel_time, dff_windows, z_windows, keep


def count_baseline_overlaps(event_times: np.ndarray, all_outcomes: np.ndarray, baseline) -> int:
    """Events whose baseline window ``[t + start, t)`` contains another outcome."""
    ordered = np.sort(all_outcomes)
    inside = np.searchsorted(ordered, event_times, side="left") - np.searchsorted(
        ordered, event_times + baseline[0], side="left"
    )
    return int((inside > 0).sum())


def real_condition_windows(
    rec,
    processed: dict,
    events: pd.DataFrame,
    condition: str,
    window: tuple[float, float],
    b_settings,
    baseline,
) -> dict:
    """Event-aligned traces of one recording and one outcome, in all views, with the counts behind them.

    *processed* is the recording's ``PROCESSED[rid]["filtered"]``, *events* its labelled outcome events, *window*
    the extraction window and *baseline* the interval checked for other outcomes.
    """
    times = events.loc[events["condition"] == condition, "outcome_time"].to_numpy()
    rel_time, dff_windows, z_windows, keep = trace_windows(rec, processed, times, window)
    views, valid = compute_views(dff_windows, z_windows, rel_time, b_settings)
    for name, trace in channel_traces(processed).items():
        views[name] = extract_event_windows(rec.time, trace, times, window, rec.fs)[1][valid]
    return {
        "rel_time": rel_time,
        "views": views,
        "n_events": len(times),
        "n_outside_sync": int((~keep).sum()),
        "n_zero_baseline_sd": int((~valid).sum()),
        "n_used": int(valid.sum()),
        "n_baseline_overlap": count_baseline_overlaps(
            times[keep][valid], events["outcome_time"].to_numpy(), baseline
        ),
    }


REAL = {
    (rid, cond): real_condition_windows(
        rec,
        PROCESSED[rid]["filtered"],
        EVENTS[rid],
        cond,
        EXTRACTION_WINDOW,
        B_SETTINGS,
        B_Z_BASELINE,
    )
    for rid, rec in RECORDINGS.items()
    for cond in CONDITIONS
}
REL_TIME = next(iter(REAL.values()))["rel_time"]

exclusions = pd.DataFrame(
    [
        {
            "recording": rid,
            "mouse": RECORDINGS[rid].subject,
            "condition": cond,
            **{k: v for k, v in r.items() if k.startswith("n_")},
        }
        for (rid, cond), r in REAL.items()
    ]
)
show_table(exclusions, "window_exclusions")
print(
    f"Trials: {exclusions.groupby('condition')['n_events'].sum().to_dict()} found, "
    f"{exclusions.groupby('condition')['n_used'].sum().to_dict()} used; "
    f"{exclusions['n_outside_sync'].sum()} excluded for incomplete windows, "
    f"{exclusions['n_zero_baseline_sd'].sum()} for a zero baseline SD; "
    f"{exclusions['n_baseline_overlap'].sum()} used trials have another outcome inside the baseline window."
)

In [ ]:
def pool_real(real: dict, recordings: dict, mouse: str, condition: str, view: str) -> np.ndarray:
    """All trials of *mouse* for one outcome and view, over its recordings (rows are trials).

    *real* maps ``(recording id, condition)`` to :func:`real_condition_windows` output; *recordings* maps id to recording.
    """
    parts = [
        real[(rid, condition)]["views"][view]
        for rid, rec in recordings.items()
        if rec.subject == mouse
    ]
    return np.concatenate(parts, axis=0)


def plot_outcome_responses(
    ax, rel_time, trials_by_condition: dict[str, np.ndarray], ylabel: str, legend: bool = True
) -> None:
    """Mean ± SEM across trials for each outcome, with n in the legend (if *legend*) and the event marked."""
    for condition, trials in trials_by_condition.items():
        label = f"{condition.capitalize()} (n = {len(trials)})"
        if len(trials) == 0:
            ax.plot([], [], color=CONDITION_COLORS[condition], label=label)
            continue
        mean, sem = mean_and_sem(trials)
        ax.plot(rel_time, mean, color=CONDITION_COLORS[condition], lw=1.3, label=label)
        ax.fill_between(
            rel_time, mean - sem, mean + sem, color=CONDITION_COLORS[condition], alpha=0.25, lw=0
        )
    ax.axvline(0, color="k", ls="--", lw=0.8)
    ax.axhline(0, color="0.8", lw=0.6)
    ax.set_ylabel(ylabel)
    if legend:
        ax.legend(loc="upper right", frameon=False)


def response_grid(
    views: tuple[str, ...],
    title: str,
    plot_cell: Callable,
    mice: list[str],
    event_window: tuple[float, float],
    legend_handles=None,
    axes=None,
):
    """One row per mouse, one column per view; *plot_cell* draws a mouse's cell into an axis.

    *legend_handles* (optional) puts one shared legend below the figure instead of one per panel. *axes* is an
    optional ``(len(mice), len(views))`` array of axes to draw on; if None a new figure is created. Returns the figure.
    """
    if axes is None:
        _, axes = plt.subplots(
            len(mice),
            len(views),
            figsize=(max(11, 3.9 * len(views)), 2.3 * len(mice) + (0.6 if legend_handles else 0)),
            sharex=True,
            constrained_layout=True,
            squeeze=False,
        )
    axes = np.asarray(axes).reshape(len(mice), len(views))
    fig = axes[0, 0].figure
    for row, mouse in zip(axes, mice, strict=True):
        for ax, view in zip(row, views, strict=True):
            plot_cell(ax, mouse, view, VIEW_LABELS[view][1])
        row[0].annotate(
            mouse,
            xy=(0.02, 0.96),
            xycoords="axes fraction",
            va="top",
            fontsize=9,
            fontweight="bold",
        )
    for ax, view in zip(axes[0], views, strict=True):
        ax.set_title(VIEW_LABELS[view][0])
    for ax in axes[-1]:
        ax.set_xlabel("Time From Outcome (s)")
        ax.set_xlim(event_window)
    if legend_handles:
        fig.legend(
            handles=legend_handles,
            loc="outside lower center",
            ncol=len(legend_handles),
            frameon=False,
        )
    fig.suptitle(title, fontsize=12)
    return fig


def plot_pooled_cell(ax, mouse, view, ylabel, *, real, recordings, rel_time):
    """Section-7 cell: rewarded and unrewarded responses of *mouse* in *view*."""
    pooled = {c: pool_real(real, recordings, mouse, c, view) for c in CONDITIONS}
    plot_outcome_responses(ax, rel_time, pooled, ylabel)


plot_pooled = partial(plot_pooled_cell, real=REAL, recordings=RECORDINGS, rel_time=REL_TIME)

### Pipeline A: session ΔF/F and session z-score

In [ ]:
fig = response_grid(
    ("A_dff", "A_z"),
    "Pipeline A: Outcome-Aligned Responses (Mean ± SEM Across Trials)",
    plot_pooled,
    MICE,
    EVENT_WINDOW,
)
save_figure(fig, "responses_pipeline_A")
plt.show()

### Pipeline B: baseline-subtracted ΔF/F and trial z-score, smoothed

In [ ]:
fig = response_grid(
    ("B_dff", "B_z"),
    "Pipeline B: Outcome-Aligned Responses (Mean ± SEM Across Trials)",
    plot_pooled,
    MICE,
    EVENT_WINDOW,
)
save_figure(fig, "responses_pipeline_B")
plt.show()

### Peak and area under the curve

Descriptive summaries of each mouse's **mean z-score trace** over the post-event window (0 to 10 s): the largest value, its
time, and the signed area under the curve (trapezoid rule). Pipeline A's are in session SD units, pipeline B's in pre-event SD
units of the smoothed trace, so the two are not comparable in size.

In [ ]:
def response_metrics_table(
    real: dict, recordings: dict, mice: list[str], rel_time: np.ndarray, window
) -> pd.DataFrame:
    """Peak, peak time and AUC (over *window*) of each mouse's mean z-score trace, by outcome and pipeline."""
    rows = []
    for mouse in mice:
        for condition in CONDITIONS:
            for view, pipeline in (("A_z", "A"), ("B_z", "B")):
                trials = pool_real(real, recordings, mouse, condition, view)
                if len(trials) == 0:
                    continue
                mean, _ = mean_and_sem(trials)
                rows.append(
                    {
                        "mouse": mouse,
                        "condition": condition,
                        "pipeline": pipeline,
                        "n_trials": len(trials),
                    }
                    | peak_and_auc(rel_time, mean, window)
                )
    return pd.DataFrame(rows)


response_metrics = response_metrics_table(REAL, RECORDINGS, MICE, REL_TIME, METRIC_WINDOW)
show_table(response_metrics, "response_peak_auc")

### 7b. Control-channel responses to the outcomes

The primer recommends looking at the control channel aligned to events: a control that is truly isosbestic for the sensor should not
respond to neural activity, and its response (if any) shows what the movement regression will subtract. Columns, all ΔF/F (%) of the
**session** (no baseline subtraction, same window, mean ± SEM across trials):

- (a) the **signal** channel ΔF/F *before* movement correction;
- (b) the **control** channel ΔF/F relative to its own bleaching fit;
- (c) the **fitted control contribution**: the part of the signal that the regression attributes to the control and subtracts, as % of the signal's
  fit (its shape is the control's, scaled by the regression slope);
- (d) the **corrected** ΔF/F, which is exactly (a) − (c).

**How to read the patterns.**

- A response in (a) and (d) but **not** in (b): a response that exists only in the signal channel, consistent with a sensor signal.
- A **same-sign** response in (a) and (b) that is mostly gone in (d): movement or haemodynamic artefact common to both channels, which the regression
  removes. The same picture would arise if the **control itself responds to norepinephrine**; then the regression subtracts real signal, and (c) shows how much.
- An **opposite-sign** response in (b) relative to (a): the 405 nm channel is probably **not isosbestic for nLightG2** (its fluorescence changes
  with the sensor's state in the opposite direction), and the control regression cannot be trusted as a movement correction.

**Not verified:** the isosbestic point of nLightG2 has not been checked here; Meg should confirm that 405 nm is isosbestic for this sensor
(and that the channels are wired as assumed).

The table gives, per mouse and outcome, the **peak absolute response in 0-2 s** of the session-ΔF/F mean trace relative to its −2 to 0 s mean, for the signal before correction
and for the control, their ratio (signal / control), and whether the two peaks have the same sign.

In [ ]:
def peak_abs_response(rel_time: np.ndarray, mean_trace: np.ndarray, pre, post) -> float:
    """Signed value of the largest |mean trace minus its *pre*-interval mean| within the *post* interval."""
    response = mean_trace - mean_trace[interval_mask(rel_time, pre)].mean()
    window = response[interval_mask(rel_time, post)]
    return float(window[np.argmax(np.abs(window))])


def control_channel_table(
    real: dict, recordings: dict, mice: list[str], rel_time: np.ndarray, pre, post
) -> pd.DataFrame:
    """Per mouse and outcome: peak response (0-2 s) of the uncorrected signal and of the control, and their ratio."""
    rows = []
    for mouse in mice:
        for condition in CONDITIONS:
            signal = pool_real(real, recordings, mouse, condition, "signal_uncorrected")
            control = pool_real(real, recordings, mouse, condition, "control")
            if len(signal) == 0:
                continue
            peak_signal = peak_abs_response(rel_time, signal.mean(axis=0), pre, post)
            peak_control = peak_abs_response(rel_time, control.mean(axis=0), pre, post)
            rows.append(
                {
                    "mouse": mouse,
                    "condition": condition,
                    "n_trials": len(signal),
                    "peak_signal_pct": peak_signal,
                    "peak_control_pct": peak_control,
                    "ratio_signal_over_control": abs(peak_signal) / abs(peak_control),
                    "same_sign": bool(np.sign(peak_signal) == np.sign(peak_control)),
                }
            )
    return pd.DataFrame(rows)


fig = response_grid(
    CHANNEL_VIEWS,
    "Control-Channel Check: Outcome-Aligned ΔF/F Before and After Movement Correction",
    plot_pooled,
    MICE,
    EVENT_WINDOW,
)
save_figure(fig, "responses_control_channel")
plt.show()

control_channel = control_channel_table(REAL, RECORDINGS, MICE, REL_TIME, EVOKED_PRE, EVOKED_POST)
show_table(control_channel, "qc_7_control_channel_peaks")

## 8. Random-time control (trial matched)

A response aligned to an outcome could also arise from the trace itself (slow drifts, movement, the normalisation). The control asks
what the same analysis gives at **times unrelated to behaviour**. For each recording and each outcome, as many random times
are drawn as that recording contributes complete-window trials. The draw is repeated 200
times; each draw gives a mean trace, so the 200 means form a **null distribution of the mean trace**. Random windows go through exactly the
same normalisation as real ones (pipeline B normalises each random window by its own baseline).

**Where random times may fall (default: the task period only).** A real outcome can only happen while the task is running, so the random
times are drawn uniformly **between the recording's first and last outcome event** (of the included phases), and only where the whole
−5 to +10 s window lies inside the sync coverage and clear of any stretch of missing sync time. The earlier version drew from the whole
sync coverage, which includes the minutes before the task starts and after it ends; the trace there (animal being placed, fibre settling,
bleaching still steep) differs from the trace during the task, so the null would be wrong in a way that has nothing to do with the outcomes.
Set `RANDOM_TIMES_WHOLE_COVERAGE = True` in the configuration to get the old behaviour back. The draws use a seeded generator
(`RANDOM_SEED`) with one stream per **recording file name and outcome**, so adding or removing another recording does not change them.

**Panels.** As in section 7, but now each outcome is drawn against its *own* matched random times: solid = real mean ± SEM
across trials; dashed = mean of the 200 random means; light band = their 2.5-97.5% range. Trials of a mouse's recordings are pooled with each
recording's trials counted in proportion to its number of events, matching the real pooling. **What to look for:** a
real mean that leaves the band after time zero (and not before it) is a response the trace does not produce by chance at random times.
Random times may fall near real outcomes; they are not excluded.

In [ ]:
def task_period(events: pd.DataFrame) -> tuple[float, float]:
    """First and last outcome time (s) of a recording: the span in which the task was running."""
    return float(events["outcome_time"].min()), float(events["outcome_time"].max())


def recording_rng(base_seed: int, rid: str, condition: str) -> np.random.Generator:
    """Random generator for one recording and outcome, keyed by names rather than by list position.

    The key is the CRC32 of the ``.ppd`` file stem and of the outcome name (stable across runs, unlike ``hash``),
    mixed with *base_seed* by ``SeedSequence``; adding or removing other recordings leaves this stream unchanged.
    """
    key = [base_seed, zlib.crc32(rid.encode()), zlib.crc32(condition.encode())]
    return np.random.default_rng(np.random.SeedSequence(key))


assert recording_rng(0, "a", "rewarded").uniform() == recording_rng(0, "a", "rewarded").uniform(), (
    "streams must repeat"
)
assert recording_rng(0, "a", "rewarded").uniform() != recording_rng(0, "b", "rewarded").uniform(), (
    "streams must differ"
)


def random_null_means(
    rec,
    processed: dict,
    n_events: int,
    rng: np.random.Generator,
    window: tuple[float, float],
    b_settings,
    n_draws: int,
    views: tuple[str, ...],
    event_span: tuple[float, float] | None,
) -> dict:
    """Means of *n_draws* random-time trial sets for one recording and outcome, by view.

    Args:
        rec: The recording; *processed* its ``PROCESSED[rid]["filtered"]``.
        n_events (int): Random times per draw (the real trial count).
        rng (np.random.Generator): Source of the draws.
        window (tuple[float, float]): Extraction window relative to each time (s).
        b_settings: Pipeline B settings, so random windows are normalised like real ones.
        n_draws (int): Number of draws.
        views (tuple[str, ...]): Views to average.
        event_span (tuple[float, float] | None): Span the random times must fall in (the task period), or None for the
            whole sync coverage.

    Returns:
        dict: ``{"means": {view: (n_draws, n_time)}, "n": events per draw}``.

    Raises:
        ValueError: If no window fits (see ``random_event_times``).
    """
    coverage, gaps = sync_coverage(rec.time), gap_spans(rec.time)
    rel_time = relative_time_grid(window, rec.fs)
    means = {view: np.empty((n_draws, len(rel_time))) for view in views}
    for draw in range(n_draws):
        times = random_event_times(rng, n_events, coverage, window, gaps, event_span)
        _, dff_windows, z_windows, _ = trace_windows(rec, processed, times, window)
        drawn, _ = compute_views(dff_windows, z_windows, rel_time, b_settings)
        for view in views:
            means[view][draw] = drawn[view].mean(axis=0)
    return {"means": means, "n": n_events}


def compute_null(recordings: dict, processed: dict, events: dict, real: dict, whole_coverage: bool):
    """The random-time null of every recording and outcome.

    Returns:
        tuple[dict, list[dict]]: The nulls by ``(recording id, condition)`` (None where a recording has no usable
        trial of that outcome or no random window fits), and one row per set with no null, with the reason.
    """
    null, unavailable = {}, []
    for rid, rec in recordings.items():
        span = None if whole_coverage else task_period(events[rid])
        for condition in CONDITIONS:
            n_events = real[(rid, condition)]["n_used"]
            null[(rid, condition)] = None
            if n_events == 0:
                continue
            rng = recording_rng(RANDOM_SEED, rid, condition)
            try:
                null[(rid, condition)] = random_null_means(
                    rec,
                    processed[rid]["filtered"],
                    n_events,
                    rng,
                    EXTRACTION_WINDOW,
                    B_SETTINGS,
                    N_RANDOM_DRAWS,
                    VIEWS,
                    span,
                )
            except ValueError as error:
                unavailable.append({"recording": rid, "condition": condition, "reason": str(error)})
    return null, unavailable


_start = time.time()
NULL, null_unavailable = compute_null(
    RECORDINGS, PROCESSED, EVENTS, REAL, RANDOM_TIMES_WHOLE_COVERAGE
)
print(
    f"Random-time control: {N_RANDOM_DRAWS} draws x {len(NULL)} recording-outcome sets in {time.time() - _start:.0f} s."
)
print(
    "Random times drawn over:",
    "the whole sync coverage" if RANDOM_TIMES_WHOLE_COVERAGE else "each recording's task period",
)
if null_unavailable:
    print("No random-time null for:")
    display(pd.DataFrame(null_unavailable))
TASK_PERIODS = pd.DataFrame(
    [
        {
            "recording": rid,
            "first_outcome_s": a,
            "last_outcome_s": b,
            "coverage_first_s": c,
            "coverage_last_s": d,
        }
        for rid, rec in RECORDINGS.items()
        for (a, b), (c, d) in [(task_period(EVENTS[rid]), sync_coverage(rec.time))]
    ]
)
show_table(TASK_PERIODS, "random_control_task_periods")

In [ ]:
def pool_null(
    null: dict, recordings: dict, mouse: str, condition: str, view: str
) -> np.ndarray | None:
    """Null distribution of a mouse's pooled mean trace, ``(n_draws, n_time)``: recordings weighted by event count.

    *null* maps ``(recording id, condition)`` to :func:`random_null_means` output (or None).
    """
    parts = [
        (null[(rid, condition)]["n"], null[(rid, condition)]["means"][view])
        for rid, rec in recordings.items()
        if rec.subject == mouse and null[(rid, condition)] is not None
    ]
    if not parts:
        return None
    weights = np.array([n for n, _ in parts], dtype=float)
    return sum(w * m for w, (_, m) in zip(weights, parts, strict=True)) / weights.sum()


def plot_against_random(
    ax, rel_time, real_by_condition: dict, null_by_condition: dict, ylabel: str
) -> None:
    """Real mean ± SEM per outcome over its random-time mean (dashed) and 2.5-97.5% band (no legend)."""
    for condition in real_by_condition:
        colour, null = CONDITION_COLORS[condition], null_by_condition[condition]
        if null is not None:
            low, high = np.percentile(null, [2.5, 97.5], axis=0)
            ax.fill_between(rel_time, low, high, color=colour, alpha=0.12, lw=0)
            ax.plot(rel_time, null.mean(axis=0), color=colour, ls="--", lw=1.0)
    plot_outcome_responses(ax, rel_time, real_by_condition, ylabel, legend=False)
    counts = ", ".join(f"{c} n = {len(t)}" for c, t in real_by_condition.items())
    ax.text(
        0.99,
        0.03,
        counts,
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=7,
        bbox={"fc": "white", "alpha": 0.7, "lw": 0},
    )


CONTROL_LEGEND = [
    plt.Line2D(
        [], [], color=CONDITION_COLORS["rewarded"], label="Rewarded: mean ± SEM across trials"
    ),
    plt.Line2D(
        [], [], color=CONDITION_COLORS["unrewarded"], label="Unrewarded: mean ± SEM across trials"
    ),
    plt.Line2D(
        [],
        [],
        color="0.4",
        ls="--",
        label="Random-time mean (same colour); band: 2.5-97.5% of 200 random means",
    ),
]


def plot_control_cell(ax, mouse, view, ylabel, *, real, null, recordings, rel_time):
    """Section-8 cell: real response against random times for both outcomes."""
    plot_against_random(
        ax,
        rel_time,
        {c: pool_real(real, recordings, mouse, c, view) for c in CONDITIONS},
        {c: pool_null(null, recordings, mouse, c, view) for c in CONDITIONS},
        ylabel,
    )


plot_control = partial(
    plot_control_cell, real=REAL, null=NULL, recordings=RECORDINGS, rel_time=REL_TIME
)

### Pipeline A against random times

In [ ]:
fig = response_grid(
    ("A_dff", "A_z"),
    "Pipeline A: Responses Against Trial-Matched Random Times",
    plot_control,
    MICE,
    EVENT_WINDOW,
    CONTROL_LEGEND,
)
save_figure(fig, "random_control_pipeline_A")
plt.show()

### Pipeline B against random times

In [ ]:
fig = response_grid(
    ("B_dff", "B_z"),
    "Pipeline B: Responses Against Trial-Matched Random Times",
    plot_control,
    MICE,
    EVENT_WINDOW,
    CONTROL_LEGEND,
)
save_figure(fig, "random_control_pipeline_B")
plt.show()

### How much of the post-event window lies outside the random band?

For each mouse, outcome and view: the fraction of samples between 0 and 10 s where the real mean lies outside the 2.5-97.5% random band,
and the same fraction for the pre-event samples (−5 to 0 s). The band is pointwise, so about 5% outside is expected by chance
for a curve with no real response; consecutive samples are strongly correlated, so these fractions are descriptive, not a test.

In [ ]:
def outside_band_table(
    real: dict,
    null: dict,
    recordings: dict,
    mice: list[str],
    rel_time: np.ndarray,
    post_window,
    views,
) -> pd.DataFrame:
    """Fraction of pre- and post-event samples where the real mean leaves the random band."""
    pre = rel_time < 0
    post = (rel_time >= post_window[0]) & (rel_time <= post_window[1])
    rows = []
    for mouse in mice:
        for condition in CONDITIONS:
            for view in views:
                band, trials = (
                    pool_null(null, recordings, mouse, condition, view),
                    pool_real(real, recordings, mouse, condition, view),
                )
                if band is None or len(trials) == 0:
                    continue
                low, high = np.percentile(band, [2.5, 97.5], axis=0)
                mean = trials.mean(axis=0)
                outside = (mean < low) | (mean > high)
                rows.append(
                    {
                        "mouse": mouse,
                        "condition": condition,
                        "view": view,
                        "n_trials": len(trials),
                        "frac_outside_pre": outside[pre].mean(),
                        "frac_outside_post": outside[post].mean(),
                    }
                )
    return pd.DataFrame(rows)


outside_band = outside_band_table(REAL, NULL, RECORDINGS, MICE, REL_TIME, METRIC_WINDOW, VIEWS)
show_table(outside_band, "random_control_outside_band")

## 8b. Quality control, continued: evoked SNR and the per-mouse ranking

**Evoked response** of a trial = mean over 0-2 s minus mean over −2-0 s. It is computed from the **rewarded** trials of each recording, in session-z units
for pipeline A and in smoothed trial-z units for pipeline B (each with its own random-time control from section 8). Two numbers describe a recording:

- **`d_prime` (the measure used for ranking).** The mean response divided by its SD across trials. It is a standardised effect size: it does **not** grow
  with the number of trials, so a recording is not favoured for having run more trials. Its sign says whether the response is a rise (+) or a dip (−).
- **`z_vs_random` (significance only).** The recording's mean response placed in the distribution of the same measure at the 200 random draws: (mean − null mean) / null SD.
  The null is a distribution of *means of n trials*, so its SD shrinks like 1/√n and z grows like √n for the same effect size. It tells whether a response
  is distinguishable from chance for this many trials, which is what `weak_evoked` (|z| below 2) flags, but it is **not** used for ranking: it would rank mice by how many trials they ran.

Recordings with fewer than `EVOKED_MIN_TRIALS` rewarded trials get no evoked SNR.

**Per mouse.** Each recording's *absolute* d′ is computed first, and the mouse's value is the **median of |d′| over its recordings**, so a rise in one session and a dip
in another do **not** cancel (a signed median of two recordings with d′ of −2 and +2 would be 0). The recordings' signs are shown next to it (for example "−/+" for a mouse with
one dip and one rise), because a mouse whose sessions disagree in sign is not responding consistently, whatever its |d′|.
The primary ranking is by the event-independent spectral SNR; the second ranking is by median |d′| (pipeline A). They answer different
questions (how clean is the trace vs. how strongly does it follow the outcome), so they need not agree.

In [ ]:
def trial_responses(
    trials: np.ndarray, rel_time: np.ndarray, post: tuple[float, float], pre: tuple[float, float]
) -> np.ndarray:
    """Per-trial evoked response: mean over the *post* interval minus mean over the *pre* interval."""
    return trials[:, interval_mask(rel_time, post)].mean(axis=1) - trials[
        :, interval_mask(rel_time, pre)
    ].mean(axis=1)


def evoked_snr(responses: np.ndarray, null_responses: np.ndarray) -> dict:
    """Mean response, its z-score against the random draws (significance), and d′ across trials (effect size)."""
    mean = responses.mean()
    return {
        "mean_response": mean,
        "null_mean": null_responses.mean(),
        "null_sd": null_responses.std(ddof=1),
        "z_vs_random": (mean - null_responses.mean()) / null_responses.std(ddof=1),
        "d_prime": mean / responses.std(ddof=1),
    }


def check_evoked_snr_on_synthetic_data() -> pd.DataFrame:
    """A known response (mean m, SD 0.5) must give d′ near m/0.5 at any trial count, while z grows with √n."""
    rng = np.random.default_rng(4)
    rows = []
    for n_trials in (50, 200):
        for mean_response in (0.5, 1.0, 2.0):
            responses = rng.normal(mean_response, 0.5, n_trials)
            null = rng.normal(0, 0.5 / np.sqrt(n_trials), N_RANDOM_DRAWS)
            out = evoked_snr(responses, null)
            expected_d = mean_response / 0.5
            expected_z = mean_response / (0.5 / np.sqrt(n_trials))
            assert abs(out["d_prime"] / expected_d - 1) < 0.25, out
            assert abs(out["z_vs_random"] / expected_z - 1) < 0.25, out
            rows.append(
                (
                    n_trials,
                    mean_response,
                    out["d_prime"],
                    expected_d,
                    out["z_vs_random"],
                    expected_z,
                )
            )
    table = pd.DataFrame(
        rows,
        columns=["n_trials", "true_mean", "d_prime", "expected_d", "z_vs_random", "expected_z"],
    )
    by_n = table.groupby("n_trials")[["d_prime", "z_vs_random"]].mean()
    assert abs(by_n.loc[200, "d_prime"] / by_n.loc[50, "d_prime"] - 1) < 0.15, (
        by_n
    )  # d′ does not depend on n
    assert by_n.loc[200, "z_vs_random"] / by_n.loc[50, "z_vs_random"] > 1.7, (
        by_n
    )  # z grows by about √4 = 2
    return table


display(check_evoked_snr_on_synthetic_data())


def evoked_row(rec, inputs: dict, rel_time: np.ndarray, post, pre, min_trials: int) -> dict:
    """Evoked SNR columns of one recording for both pipelines.

    *inputs* holds the recording's rewarded data: ``n_used`` trials, the ``views`` of the real trials and the ``null``
    (as returned by :func:`random_null_means`, or None).
    """
    row = {"n_rewarded_trials": inputs["n_used"]}
    for view, name in (("A_z", "a"), ("B_z", "b")):
        keys = ("mean_response", "z_vs_random", "d_prime")
        if inputs["n_used"] < min_trials or inputs["null"] is None:
            row |= {f"{name}_{k}": np.nan for k in keys}
            continue
        null_responses = trial_responses(
            inputs["null"]["means"][view], rel_time, post, pre
        )  # a mean trace's response = mean of trial responses
        out = evoked_snr(
            trial_responses(inputs["views"][view], rel_time, post, pre), null_responses
        )
        row |= {f"{name}_{k}": out[k] for k in keys}
    return row


def evoked_qc(
    recordings: dict, real: dict, null: dict, thresholds: dict, rel_time, post, pre
) -> pd.DataFrame:
    """Evoked-SNR table for every recording, with flags."""
    inputs = {
        rid: {
            "n_used": real[(rid, "rewarded")]["n_used"],
            "views": real[(rid, "rewarded")]["views"],
            "null": null[(rid, "rewarded")],
        }
        for rid in recordings
    }
    table = per_recording(
        recordings,
        evoked_row,
        inputs,
        rel_time=rel_time,
        post=post,
        pre=pre,
        min_trials=thresholds["min_evoked_trials"],
    )
    return attach_flags(
        table,
        {
            "few_trials": table["n_rewarded_trials"] < thresholds["min_evoked_trials"],
            "weak_evoked": table["a_z_vs_random"].abs() < thresholds["min_abs_evoked_z"],
        },
    )


qc_evoked = evoked_qc(RECORDINGS, REAL, NULL, QC_THRESHOLDS, REL_TIME, EVOKED_POST, EVOKED_PRE)
show_table(qc_evoked, "qc_6_evoked")

In [ ]:
def split_flags(flags: pd.Series) -> list[str]:
    """Flag names from a column of comma-separated flag strings."""
    return [f for cell in flags.fillna("") for f in cell.split(", ") if f]


def sign_string(values: pd.Series) -> str:
    """The signs of the finite *values* in order, e.g. ``"−/+"`` (empty if none is finite)."""
    return "/".join("−" if v < 0 else "+" for v in values if np.isfinite(v))


def evoked_by_mouse(evoked: pd.DataFrame) -> pd.DataFrame:
    """Per mouse: median over recordings of |d′| (the ranking basis), the recordings' signs, and median |z| (significance).

    The median is taken of the absolute values, so recordings with opposite-sign responses do not cancel.
    """
    rows = []
    for mouse, group in evoked.groupby("mouse", sort=False):
        rows.append(
            {
                "mouse": mouse,
                "abs_d_prime_a": group["a_d_prime"].abs().median(),
                "signs_a": sign_string(group["a_d_prime"]),
                "abs_d_prime_b": group["b_d_prime"].abs().median(),
                "signs_b": sign_string(group["b_d_prime"]),
                "median_abs_z_a": group["a_z_vs_random"].abs().median(),
            }
        )
    return pd.DataFrame(rows)


def check_evoked_by_mouse_on_synthetic_data() -> pd.DataFrame:
    """Opposite-sign recordings must not cancel, and a mouse with more trials must not outrank one with a stronger effect."""
    evoked = pd.DataFrame(
        {
            "mouse": ["M1", "M1", "M2", "M2", "M3"],
            "a_d_prime": [-2.0, 2.0, 0.5, 0.5, 1.0],
            "b_d_prime": [-1.0, 1.0, 0.2, 0.2, 0.4],
            "a_z_vs_random": [-8.0, 8.0, 12.0, 12.0, 3.0],  # M2 has many trials: big z, small d′
        }
    )
    out = evoked_by_mouse(evoked).set_index("mouse")
    assert out.loc["M1", "abs_d_prime_a"] == 2.0 and out.loc["M1", "signs_a"] == "−/+", out
    assert out.loc["M2", "signs_a"] == "+/+" and out.loc["M3", "signs_a"] == "+", out
    ranks = out["abs_d_prime_a"].rank(ascending=False, method="min")
    assert ranks.to_dict() == {"M1": 1.0, "M2": 3.0, "M3": 2.0}, (
        ranks
    )  # ranked by d′, not by M2's large z
    return out.reset_index()


display(check_evoked_by_mouse_on_synthetic_data())


def mouse_summary(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Per mouse: median SNR measures over its recordings, recording count, flag counts and both ranks."""
    snr, evoked = tables["snr"], tables["evoked"]
    rows = []
    for mouse, group in snr.groupby("mouse", sort=False):
        flags = pd.Series(
            split_flags(pd.concat([t.loc[t["mouse"] == mouse, "flags"] for t in tables.values()]))
        )
        rows.append(
            {
                "mouse": mouse,
                "n_recordings": len(group),
                "snr_db": group["snr_db"].median(),
                "specificity_db": group["specificity_db"].median(),
                "flags": ", ".join(f"{k} x{v}" for k, v in flags.value_counts().items()),
            }
        )
    out = pd.DataFrame(rows).merge(evoked_by_mouse(evoked), on="mouse")
    out["rank_snr"] = out["snr_db"].rank(ascending=False, method="min").astype(int)
    out["rank_evoked"] = (
        out["abs_d_prime_a"].rank(ascending=False, method="min", na_option="bottom").astype(int)
    )
    return out


MOUSE_SUMMARY = mouse_summary(
    {
        "acquisition": qc_acquisition,
        "denoising": qc_denoising,
        "bleaching": qc_bleaching,
        "movement": qc_movement,
        "snr": qc_snr,
        "evoked": qc_evoked,
    }
)
_rewarded_peaks = control_channel.query("condition == 'rewarded'").set_index("mouse")[
    ["peak_signal_pct", "peak_control_pct", "ratio_signal_over_control"]
]
MOUSE_SUMMARY = MOUSE_SUMMARY.merge(
    _rewarded_peaks.add_suffix("_rewarded"), left_on="mouse", right_index=True, how="left"
)
save_table(MOUSE_SUMMARY, "qc_mouse_summary")
RANKING_COLUMNS = [
    "mouse",
    "n_recordings",
    "snr_db",
    "specificity_db",
    "abs_d_prime_a",
    "signs_a",
    "flags",
]

print(
    "Mice ranked by primary SNR (spectral, event-independent); median over each mouse's recordings:\n"
)
by_snr = MOUSE_SUMMARY.sort_values("rank_snr")[["rank_snr", *RANKING_COLUMNS]].rename(
    columns={"rank_snr": "rank"}
)
print(by_snr.to_string(index=False))

print(
    "\nPeak |response| 0-2 s of the uncorrected signal and of the control (session ΔF/F %), rewarded trials:\n"
)
print(
    MOUSE_SUMMARY[
        [
            "mouse",
            "peak_signal_pct_rewarded",
            "peak_control_pct_rewarded",
            "ratio_signal_over_control_rewarded",
        ]
    ].to_string(index=False)
)
print(
    "\nMice ranked by evoked d′ (median over recordings of |d′| of the rewarded response, pipeline A):\n"
)
by_evoked = MOUSE_SUMMARY.sort_values("rank_evoked")[
    [
        "rank_evoked",
        "mouse",
        "n_recordings",
        "abs_d_prime_a",
        "signs_a",
        "abs_d_prime_b",
        "signs_b",
        "median_abs_z_a",
        "snr_db",
    ]
]
print(
    "(Ranked by |d′|, which does not depend on the trial count. signs_a: sign of each recording's d′, '−/+' = one dip and one rise."
)
print(
    " median_abs_z_a, the z against random times, is shown for significance only and is not used for the ranking.)"
)
print(by_evoked.rename(columns={"rank_evoked": "rank"}).to_string(index=False))

gap = (MOUSE_SUMMARY["rank_snr"] - MOUSE_SUMMARY["rank_evoked"]).abs()
disagree = MOUSE_SUMMARY.loc[gap >= MIN_RANK_DISAGREEMENT, ["mouse", "rank_snr", "rank_evoked"]]
print(f"\nMice whose two ranks differ by {MIN_RANK_DISAGREEMENT} or more:")
print(disagree.to_string(index=False) if len(disagree) else "  none")
print(
    f"Spearman correlation between the two rankings: {MOUSE_SUMMARY['rank_snr'].corr(MOUSE_SUMMARY['rank_evoked'], method='spearman'):.2f}"
)
mixed = MOUSE_SUMMARY.loc[
    MOUSE_SUMMARY["signs_a"].str.contains("−") & MOUSE_SUMMARY["signs_a"].str.contains(r"\+"),
    ["mouse", "signs_a"],
]
print(
    "Mice whose recordings disagree in the sign of the evoked response:",
    mixed.to_string(index=False) if len(mixed) else "none",
)

In [ ]:
def plot_mouse_ranking(axes, summary: pd.DataFrame) -> None:
    """Horizontal bars of primary SNR, specificity and evoked |d′| per mouse, best primary SNR at the top.

    The evoked panel writes each mouse's per-recording signs beside its bar.
    """
    ordered = summary.sort_values("rank_snr", ascending=False)
    for ax, column, label in zip(
        axes,
        ("snr_db", "specificity_db", "abs_d_prime_a"),
        (
            "Spectral SNR (dB)",
            "Specificity: Uncorrected Sensor Minus Control (dB)",
            "Evoked |d′| (Median Over Recordings; Signs Shown)",
        ),
        strict=True,
    ):
        ax.barh(ordered["mouse"], ordered[column], color="tab:blue")
        ax.axvline(0, color="k", lw=0.6)
        ax.set(xlabel=label, title=label.split(" (")[0])
    for y, (value, signs) in enumerate(
        zip(ordered["abs_d_prime_a"], ordered["signs_a"], strict=True)
    ):
        axes[2].annotate(
            signs, xy=(value, y), xytext=(3, 0), textcoords="offset points", va="center", fontsize=8
        )
    axes[2].margins(x=0.2)
    axes[0].set_ylabel("Mouse (Best Primary SNR at Top)")


fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True, constrained_layout=True)
plot_mouse_ranking(axes, MOUSE_SUMMARY)
fig.suptitle("Per-Mouse SNR (Median Over Recordings)", fontsize=12)
save_figure(fig, "qc_mouse_ranking")
plt.show()

## 9. Summary

**What was run.** Cohort-04 recordings were put on pyControl time (photometry and video, section 2), preprocessed by pipeline A (primer
order) and pipeline B (nLightG2 event-level normalisation, reconstructed from the earlier notebook's methods excerpt), checked
with the QC suite (section 6 and 8b), aligned to rewarded and unrewarded outcomes (section 7) and compared with trial-matched
random times (section 8). The counts and locations are printed below.

**Observations from this run** (see the tables and figures for the numbers):

- Sync is good for all recordings (matched fraction near 1, interval disagreement within one photometry sample); video lies on the same clock.
- The detrended signal and control are strongly correlated in most recordings (R² in `qc_4_movement`), so most of the signal's
  variance is shared with the control and is removed by step A5.
- Uncorrected specificity (uncorrected sensor SNR minus control SNR, `specificity_db` in `qc_5_snr`) has per-mouse medians from **−4.7 to +3.6 dB**:
  within ±1 dB for 5 of 10 mice (RT_64_N −0.25, RT_177_N +0.68, RT_176_N −0.90, RT_177_LR −0.01, RT_64_R −0.07), **below −1 dB for 4** (RT_177_R −3.18,
  RT_177_L −2.02, RT_176_R −4.65, RT_65_LR −1.93: the control carries *more* slow power than the sensor) and above +1 dB for only 1 (RT_176_L +3.59).
  13 of 17 recordings fall below the 1-dB flag. So slow fluctuations are about as strong in the control as in the signal channel, or stronger.
- The control-channel check (section 7b, `qc_7_control_channel_peaks`) shows the outcome-locked response in the *control* too, with the **same sign** as in
  the uncorrected signal and a size comparable to the signal's (signal/control peak ratio 0.4-2, mostly 0.7-1). The regression removes most of it, leaving responses of
  order 0.05-0.1 % ΔF/F. A same-sign response in both channels fits movement/haemodynamic artefact, or a control that responds to the sensor's target;
  an opposite sign would have indicated a non-isosbestic control (not seen).
- Units were verified on the example recording (SD of the corrected signal 6 mV on a 2.6 V baseline = 0.23 %); the movement regression removes 15-94 % of the SD of the
  session ΔF/F (`qc_correction_size`), so the small corrected ΔF/F reflects how much of the signal is shared with the control.
- The LED-off baseline contains brief upward spikes in some sessions (`*_baseline_spikes_per_min` in `qc_1_acquisition`); in the example
  recording they are not at the rsync pulses. Their origin is not known.
- **Evoked ranking.** Mice are ranked by the median over their recordings of |d′| (mean rewarded response / SD across trials; it does not grow with the trial count),
  with each recording's sign shown. Three mice have recordings that disagree in sign (RT_64_N "+/−/−", RT_64_R "−/+", RT_177_N "+/+/−"); under the earlier signed-z median
  these cancelled (RT_64_R: −2.56 and +4.61 gave 1.02). The z against random times is kept for significance only, because it grows with √n.
  The two rankings (primary spectral SNR, median |d′|) are only weakly related in this cohort (Spearman about 0.1); they measure different things, and
  with 1-4 recordings per mouse neither ranking is stable.
- **Random-time control.** Random times are now confined to each recording's task period (first to last outcome) and to windows clear of sync gaps. The fraction of
  pre-event samples outside the random band (`random_control_outside_band`) should be near 5% if the null describes the baseline; it is far above that for several
  cells (for example pipeline A, rewarded: RT_177_N 0.54, RT_177_LR 0.54, RT_177_L 0.39; unrewarded RT_176_N 0.63), where the real pre-event trace is itself unlike the
  random-time distribution, so a post-event excursion outside the band is not interpretable on its own there.

**Open questions.**

1. **Channel wiring.** `analog_1` = nLightG2 (470 nm) and `analog_2` = isosbestic control (405 nm) is assumed from the earlier notebooks and has not been
   confirmed against the physical LED connections.
2. **nLightG2 methods.** Pipeline B follows the methods excerpt adapted in the earlier notebook, not the paper's text. Please check
   the baseline windows (paper: 20 s and 15 s), the z-score definition, the smoothing length and the peak/AUC definitions.
3. **The 6-ms cue offset.** The no-reward cue timer fires 494 ms after cue onset (logged 500 ms), below one 7.7-ms photometry sample. Should
   unrewarded events be shifted? (Its cause is not established.)
4. **Sharp peak or dip at t = 0 in both outcomes.** Several mice show a sharp deflection at the outcome for rewarded *and* unrewarded trials,
   present in the control channel as well (section 7b). Is it physiological, or task-locked (light, sound, pump, a reflection when the animal moves at the port)?
5. **Uncorrected specificity at or below 0 dB.** For 9 of 10 mice slow fluctuations are as strong in the control as in the signal, or stronger (median −4.7 to +0.7 dB). Is 405 nm isosbestic
   for nLightG2 (not verified; please check), and is the signal channel dominated by movement/haemodynamics in these recordings?
6. **Small ΔF/F.** Corrected responses are about 0.05-0.1 % ΔF/F, small for an NE sensor. The arithmetic is verified (section 3); the open question
   is whether sensor expression/fibre placement, the large autofluorescence-containing denominator (about 2.6 V), or the regression removing real signal explains it.
7. **Thresholds.** The QC thresholds are starting points (`low_specificity` below 1 dB, `weak_evoked` below |z| = 2; the z flag is a significance screen, not the ranking).
8. **Recordings of concern.** The 1.9-minute recording of RT_177_N (ses-40, one trial) is flagged `short` and `outside_sync` and contributes almost
   nothing; several mice have only one session, so their medians rest on one recording.

In [ ]:
n_used = exclusions.groupby("condition")["n_used"].sum()
print(f"Mice: {len(MICE)}; recordings: {len(RECORDINGS)}; skipped .ppd files: {len(skipped)}.")
print(f"Trials used: {int(n_used.sum())} ({n_used.to_dict()}); phases: {PHASES}.")
print(
    f"Recordings flagged low_specificity (< {QC_THRESHOLDS['min_specificity_db']} dB): "
    f"{int((qc_snr['specificity_db'] < QC_THRESHOLDS['min_specificity_db']).sum())} of {len(qc_snr)}."
)
print(f"Figures: {FIGURE_DIR.relative_to(REPO_ROOT)}")
print(f"Tables:  {TABLE_DIR.relative_to(REPO_ROOT)}")
print(f"Cache:   {memory.location}")